[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/09-system-design/04_databases.ipynb)

*System Design · Notebook 4 of 10*

# Databases, Replication & Sharding

Choosing a database, indexes and the leftmost-prefix rule, LSM trees and compaction, replication lag, quorums (R + W &gt; N) and read repair, hash vs range sharding, shard keys, hot shards, scatter-gather, joins, N+1 queries, lost updates and optimistic locking, connection pools and read replicas.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Sharding is splitting a huge phone book into volumes by surname: each volume is quick to search, but a question across all names means opening every volume.

**What interviewers really test here:** Choosing a database, indexes, LSM vs B-tree, replication lag, quorums, shard keys, hot shards, lost updates and connection pools.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Choosing a database *(Basics)*
- [ ] 2. Indexes and storage engines *(Core)*
- [ ] 3. Replication *(Core)*
- [ ] 4. Sharding (partitioning) *(Core)*
- [ ] 5. Transactions and concurrency *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### ACID vs BASE

| ACID (most relational) | BASE (many NoSQL) |
| --- | --- |
| Atomic, Consistent, Isolated, Durable | Basically Available, Soft state, Eventually consistent |
| correctness first | availability and scale first |

### Isolation levels

| Level | Prevents |
| --- | --- |
| Read committed | dirty reads |
| Repeatable read / snapshot | non-repeatable reads |
| Serializable | all anomalies, incl. write skew |

---
# Part 2: Lessons, step by step

## Lesson 1: Choosing a database

*Level: Basics*

| Type | Examples | Pick it for |
| --- | --- | --- |
| Relational | PostgreSQL, MySQL | transactions, joins, constraints — the default |
| Key-value | Redis, DynamoDB | fast lookups by key, sessions, carts |
| Document | MongoDB | flexible, nested records |
| Wide-column | Cassandra, Bigtable | huge write volumes, time-ordered data |
| Graph | Neo4j | many-hop relationships |
| Search | Elasticsearch, OpenSearch | full-text and faceted search |
| Time-series | TimescaleDB, Prometheus | metrics over time |



In an interview, start relational unless a requirement pushes you elsewhere — and say which requirement.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q7](#q7), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 2: Indexes and storage engines

*Level: Core*

An **index** is a sorted structure (usually a B-tree) that finds rows in O(log n) instead of scanning the table. A composite index on `(country, city, age)` helps queries that filter from the left: `country`, or `country + city` — not `city` alone.



**B-trees** update pages in place: great reads, steady writes (PostgreSQL, MySQL). **LSM trees** append writes to an in-memory table, flush sorted files (SSTables) and merge them in the background (compaction): very fast writes, reads may check several files — Bloom filters help skip them (Cassandra, RocksDB).

**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q12](#q12), [Q13](#q13), [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 3: Replication

*Level: Core*

Copies of the data give read scale and survive failures.




- **Leader-follower**: one primary takes writes, replicas serve reads. Asynchronous replication means replicas lag — a user may not see their own update. Fixes: read your own writes from the primary, or wait until the replica has caught up.
- **Multi-leader**: writes in several regions, conflicts must be resolved.
- **Leaderless** (Dynamo style): write to W of N replicas, read from R. If `R + W > N` the read and write sets overlap, so a read sees the latest write; stale replicas are fixed by read repair.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q3](#q3), [Q4](#q4), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 4: Sharding (partitioning)

*Level: Core*

When one machine can't hold the data or the writes, split rows across shards by a **shard key**.




- **Hash sharding** spreads keys evenly but range queries hit every shard.
- **Range sharding** keeps neighbours together (good for scans) but sequential keys like timestamps all land on the newest shard.
- A good shard key has many distinct values, even traffic, and appears in most queries — otherwise queries must **scatter-gather** across all shards.
- Resharding: split big shards, use consistent hashing or many small virtual shards so data moves in small pieces.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import bisect
import zlib
from collections import Counter


def shard_demo():
    keys = [f"order:{i}" for i in range(12)]
    print("hash :", Counter(zlib.crc32(k.encode()) % 3 for k in keys))
    times = list(range(1000, 1012))
    print("range:", Counter(bisect.bisect_right([500, 900], t) for t in times), "← all new writes on the last shard")


shard_demo()

hash : Counter({1: 7, 0: 3, 2: 2})
range: Counter({2: 12}) ← all new writes on the last shard


**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q16](#q16), [Q17](#q17), [Q18](#q18), [Q6](#q6), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 5: Transactions and concurrency

*Level: Advanced*

Two requests doing read → modify → write on the same row can overwrite each other (the **lost update**). Fixes:




- Atomic updates: `UPDATE account SET bal = bal - 30 WHERE id = 7`.
- Pessimistic locks: `SELECT ... FOR UPDATE`.
- Optimistic locking: keep a version column, `UPDATE ... WHERE version = 4`, retry if 0 rows changed.



Size connection pools with Little's law (QPS × query time) and remember the database's own connection limit — a pooler such as PgBouncer sits in between at scale.

**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q9](#q9), [Q10](#q10). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
from collections import ChainMap, Counter, defaultdict

import zlib


<a id="q1"></a>
### Q1. Pick a Database

🟢 Warm-up · *Match needs to a data model*

For each workload (a list of needs), return the first matching database type using the priority order in `RULES`, or 'relational' when nothing matches — a relational database is the safe default.

**Examples**

- `workloads = [['transactions', 'reports'], ['key lookups', 'sessions'], ['relationships', 'recommendations'], ['massive writes', 'key lookups'], ['full-text', 'flexible schema'], ['reports']]` → `['relational', 'key-value', 'graph', 'wide-column', 'search engine', 'relational']`

In [ ]:
RULES = [("relationships", "graph"), ("full-text", "search engine"), ("metrics", "time-series"),
         ("transactions", "relational"), ("flexible schema", "document"),
         ("massive writes", "wide-column"), ("key lookups", "key-value")]

def pick_db(workloads):
    """
    For each workload (a list of needs), return the first matching database type
    using the priority order in RULES, or 'relational' when nothing matches — a
    relational database is the safe default.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pick_db, [
    (([['transactions', 'reports'], ['key lookups', 'sessions'], ['relationships', 'recommendations'], ['massive writes', 'key lookups'], ['full-text', 'flexible schema'], ['reports']],), ['relational', 'key-value', 'graph', 'wide-column', 'search engine', 'relational']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/elif chain per workload.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`next(...)` over the rule table.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if/elif chain per workload.

```python
def pick_db(workloads):
    out = []
    for needs in workloads:
        if "relationships" in needs:
            out.append("graph")
        elif "full-text" in needs:
            out.append("search engine")
        elif "metrics" in needs:
            out.append("time-series")
        elif "transactions" in needs:
            out.append("relational")
        elif "flexible schema" in needs:
            out.append("document")
        elif "massive writes" in needs:
            out.append("wide-column")
        elif "key lookups" in needs:
            out.append("key-value")
        else:
            out.append("relational")
    return out
```

**Cost:** O(w · r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `next(...)` over the rule table.

```python
def pick_db(workloads):
    return [next((db for need, db in RULES if need in needs), "relational") for needs in workloads]
```

**Cost:** O(w · r) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/elif chain per workload. That costs O(w · r) time and O(1) space."

"I can do better: `next(...)` over the rule table. That's O(w · r) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `workloads = [['transactions'], ['key lookups']]`

1. GeneratorExit raised: 
2. GeneratorExit raised: 
3. pick\_db() returns ['relational', 'key-value']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/pick-a-database)

<a id="q2"></a>
### Q2. What an Index Saves

🟢 Warm-up · *Rows examined*

Count rows examined to find all rows where `field == value`: a full scan reads every row; an index (sorted keys) costs about ⌈log₂ n⌉ steps plus one per match. Return `(scan, indexed)` for each query.

**Examples**

- `rows = [{'id': 0, 'city': 'Pune'}, {'id': 1, 'city': 'Delhi'}, {'id': 2, 'city': 'Goa'}, {'id': 3, 'city': 'Pune'}, {'id': 4, 'city': 'Pune'}, {'id': 5, 'city': 'Delhi'}, {'id': 6, 'city': 'Goa'}, {'id': 7, 'city': 'Pune'}, {'id': 8, 'city': 'Pune'}, {'id': 9, 'city': 'Delhi'}, {'id': 10, 'city': 'Goa'}, {'id': 11, 'city': 'Pune'}, {'id': 12, 'city': 'Pune'}, {'id': 13, 'city': 'Delhi'}, {'id': 14, 'city': 'Goa'}, {'id': 15, 'city': 'Pune'}, {'id': 16, 'city': 'Pune'}, {'id': 17, 'city': 'Delhi'}, {'id': 18, 'city': 'Goa'}, {'id': 19, 'city': 'Pune'}, {'id': 20, 'city': 'Pune'}, {'id': 21, 'city': 'Delhi'}, {'id': 22, 'city': 'Goa'}, {'id': 23, 'city': 'Pune'}, {'id': 24, 'city': 'Pune'}, {'id': 25, 'city': 'Delhi'}, {'id': 26, 'city': 'Goa'}, {'id': 27, 'city': 'Pune'}, {'id': 28, 'city': 'Pune'}, {'id': 29, 'city': 'Delhi'}, {'id': 30, 'city': 'Goa'}, {'id': 31, 'city': 'Pune'}, {'id': 32, 'city': 'Pune'}, {'id': 33, 'city': 'Delhi'}, {'id': 34, 'city': 'Goa'}, {'id': 35, 'city': 'Pune'}, {'id': 36, 'city': 'Pune'}, {'id': 37, 'city': 'Delhi'}, {'id': 38, 'city': 'Goa'}, {'id': 39, 'city': 'Pune'}, {'id': 40, 'city': 'Pune'}, {'id': 41, 'city': 'Delhi'}, {'id': 42, 'city': 'Goa'}, {'id': 43, 'city': 'Pune'}, {'id': 44, 'city': 'Pune'}, {'id': 45, 'city': 'Delhi'}, {'id': 46, 'city': 'Goa'}, {'id': 47, 'city': 'Pune'}, {'id': 48, 'city': 'Pune'}, {'id': 49, 'city': 'Delhi'}, {'id': 50, 'city': 'Goa'}, {'id': 51, 'city': 'Pune'}, {'id': 52, 'city': 'Pune'}, {'id': 53, 'city': 'Delhi'}, {'id': 54, 'city': 'Goa'}, {'id': 55, 'city': 'Pune'}, {'id': 56, 'city': 'Pune'}, {'id': 57, 'city': 'Delhi'}, {'id': 58, 'city': 'Goa'}, {'id': 59, 'city': 'Pune'}, {'id': 60, 'city': 'Pune'}, {'id': 61, 'city': 'Delhi'}, {'id': 62, 'city': 'Goa'}, {'id': 63, 'city': 'Pune'}, {'id': 64, 'city': 'Pune'}, {'id': 65, 'city': 'Delhi'}, {'id': 66, 'city': 'Goa'}, {'id': 67, 'city': 'Pune'}, {'id': 68, 'city': 'Pune'}, {'id': 69, 'city': 'Delhi'}, {'id': 70, 'city': 'Goa'}, {'id': 71, 'city': 'Pune'}, {'id': 72, 'city': 'Pune'}, {'id': 73, 'city': 'Delhi'}, {'id': 74, 'city': 'Goa'}, {'id': 75, 'city': 'Pune'}, {'id': 76, 'city': 'Pune'}, {'id': 77, 'city': 'Delhi'}, {'id': 78, 'city': 'Goa'}, {'id': 79, 'city': 'Pune'}, {'id': 80, 'city': 'Pune'}, {'id': 81, 'city': 'Delhi'}, {'id': 82, 'city': 'Goa'}, {'id': 83, 'city': 'Pune'}, {'id': 84, 'city': 'Pune'}, {'id': 85, 'city': 'Delhi'}, {'id': 86, 'city': 'Goa'}, {'id': 87, 'city': 'Pune'}, {'id': 88, 'city': 'Pune'}, {'id': 89, 'city': 'Delhi'}, {'id': 90, 'city': 'Goa'}, {'id': 91, 'city': 'Pune'}, {'id': 92, 'city': 'Pune'}, {'id': 93, 'city': 'Delhi'}, {'id': 94, 'city': 'Goa'}, {'id': 95, 'city': 'Pune'}, {'id': 96, 'city': 'Pune'}, {'id': 97, 'city': 'Delhi'}, {'id': 98, 'city': 'Goa'}, {'id': 99, 'city': 'Pune'}, {'id': 100, 'city': 'Pune'}, {'id': 101, 'city': 'Delhi'}, {'id': 102, 'city': 'Goa'}, {'id': 103, 'city': 'Pune'}, {'id': 104, 'city': 'Pune'}, {'id': 105, 'city': 'Delhi'}, {'id': 106, 'city': 'Goa'}, {'id': 107, 'city': 'Pune'}, {'id': 108, 'city': 'Pune'}, {'id': 109, 'city': 'Delhi'}, {'id': 110, 'city': 'Goa'}, {'id': 111, 'city': 'Pune'}, {'id': 112, 'city': 'Pune'}, {'id': 113, 'city': 'Delhi'}, {'id': 114, 'city': 'Goa'}, {'id': 115, 'city': 'Pune'}, {'id': 116, 'city': 'Pune'}, {'id': 117, 'city': 'Delhi'}, {'id': 118, 'city': 'Goa'}, {'id': 119, 'city': 'Pune'}, {'id': 120, 'city': 'Pune'}, {'id': 121, 'city': 'Delhi'}, {'id': 122, 'city': 'Goa'}, {'id': 123, 'city': 'Pune'}, {'id': 124, 'city': 'Pune'}, {'id': 125, 'city': 'Delhi'}, {'id': 126, 'city': 'Goa'}, {'id': 127, 'city': 'Pune'}, {'id': 128, 'city': 'Pune'}, {'id': 129, 'city': 'Delhi'}, {'id': 130, 'city': 'Goa'}, {'id': 131, 'city': 'Pune'}, {'id': 132, 'city': 'Pune'}, {'id': 133, 'city': 'Delhi'}, {'id': 134, 'city': 'Goa'}, {'id': 135, 'city': 'Pune'}, {'id': 136, 'city': 'Pune'}, {'id': 137, 'city': 'Delhi'}, {'id': 138, 'city': 'Goa'}, {'id': 139, 'city': 'Pune'}, {'id': 140, 'city': 'Pune'}, {'id': 141, 'city': 'Delhi'}, {'id': 142, 'city': 'Goa'}, {'id': 143, 'city': 'Pune'}, {'id': 144, 'city': 'Pune'}, {'id': 145, 'city': 'Delhi'}, {'id': 146, 'city': 'Goa'}, {'id': 147, 'city': 'Pune'}, {'id': 148, 'city': 'Pune'}, {'id': 149, 'city': 'Delhi'}, {'id': 150, 'city': 'Goa'}, {'id': 151, 'city': 'Pune'}, {'id': 152, 'city': 'Pune'}, {'id': 153, 'city': 'Delhi'}, {'id': 154, 'city': 'Goa'}, {'id': 155, 'city': 'Pune'}, {'id': 156, 'city': 'Pune'}, {'id': 157, 'city': 'Delhi'}, {'id': 158, 'city': 'Goa'}, {'id': 159, 'city': 'Pune'}, {'id': 160, 'city': 'Pune'}, {'id': 161, 'city': 'Delhi'}, {'id': 162, 'city': 'Goa'}, {'id': 163, 'city': 'Pune'}, {'id': 164, 'city': 'Pune'}, {'id': 165, 'city': 'Delhi'}, {'id': 166, 'city': 'Goa'}, {'id': 167, 'city': 'Pune'}, {'id': 168, 'city': 'Pune'}, {'id': 169, 'city': 'Delhi'}, {'id': 170, 'city': 'Goa'}, {'id': 171, 'city': 'Pune'}, {'id': 172, 'city': 'Pune'}, {'id': 173, 'city': 'Delhi'}, {'id': 174, 'city': 'Goa'}, {'id': 175, 'city': 'Pune'}, {'id': 176, 'city': 'Pune'}, {'id': 177, 'city': 'Delhi'}, {'id': 178, 'city': 'Goa'}, {'id': 179, 'city': 'Pune'}, {'id': 180, 'city': 'Pune'}, {'id': 181, 'city': 'Delhi'}, {'id': 182, 'city': 'Goa'}, {'id': 183, 'city': 'Pune'}, {'id': 184, 'city': 'Pune'}, {'id': 185, 'city': 'Delhi'}, {'id': 186, 'city': 'Goa'}, {'id': 187, 'city': 'Pune'}, {'id': 188, 'city': 'Pune'}, {'id': 189, 'city': 'Delhi'}, {'id': 190, 'city': 'Goa'}, {'id': 191, 'city': 'Pune'}, {'id': 192, 'city': 'Pune'}, {'id': 193, 'city': 'Delhi'}, {'id': 194, 'city': 'Goa'}, {'id': 195, 'city': 'Pune'}, {'id': 196, 'city': 'Pune'}, {'id': 197, 'city': 'Delhi'}, {'id': 198, 'city': 'Goa'}, {'id': 199, 'city': 'Pune'}, {'id': 200, 'city': 'Pune'}, {'id': 201, 'city': 'Delhi'}, {'id': 202, 'city': 'Goa'}, {'id': 203, 'city': 'Pune'}, {'id': 204, 'city': 'Pune'}, {'id': 205, 'city': 'Delhi'}, {'id': 206, 'city': 'Goa'}, {'id': 207, 'city': 'Pune'}, {'id': 208, 'city': 'Pune'}, {'id': 209, 'city': 'Delhi'}, {'id': 210, 'city': 'Goa'}, {'id': 211, 'city': 'Pune'}, {'id': 212, 'city': 'Pune'}, {'id': 213, 'city': 'Delhi'}, {'id': 214, 'city': 'Goa'}, {'id': 215, 'city': 'Pune'}, {'id': 216, 'city': 'Pune'}, {'id': 217, 'city': 'Delhi'}, {'id': 218, 'city': 'Goa'}, {'id': 219, 'city': 'Pune'}, {'id': 220, 'city': 'Pune'}, {'id': 221, 'city': 'Delhi'}, {'id': 222, 'city': 'Goa'}, {'id': 223, 'city': 'Pune'}, {'id': 224, 'city': 'Pune'}, {'id': 225, 'city': 'Delhi'}, {'id': 226, 'city': 'Goa'}, {'id': 227, 'city': 'Pune'}, {'id': 228, 'city': 'Pune'}, {'id': 229, 'city': 'Delhi'}, {'id': 230, 'city': 'Goa'}, {'id': 231, 'city': 'Pune'}, {'id': 232, 'city': 'Pune'}, {'id': 233, 'city': 'Delhi'}, {'id': 234, 'city': 'Goa'}, {'id': 235, 'city': 'Pune'}, {'id': 236, 'city': 'Pune'}, {'id': 237, 'city': 'Delhi'}, {'id': 238, 'city': 'Goa'}, {'id': 239, 'city': 'Pune'}, {'id': 240, 'city': 'Pune'}, {'id': 241, 'city': 'Delhi'}, {'id': 242, 'city': 'Goa'}, {'id': 243, 'city': 'Pune'}, {'id': 244, 'city': 'Pune'}, {'id': 245, 'city': 'Delhi'}, {'id': 246, 'city': 'Goa'}, {'id': 247, 'city': 'Pune'}, {'id': 248, 'city': 'Pune'}, {'id': 249, 'city': 'Delhi'}, {'id': 250, 'city': 'Goa'}, {'id': 251, 'city': 'Pune'}, {'id': 252, 'city': 'Pune'}, {'id': 253, 'city': 'Delhi'}, {'id': 254, 'city': 'Goa'}, {'id': 255, 'city': 'Pune'}, {'id': 256, 'city': 'Pune'}, {'id': 257, 'city': 'Delhi'}, {'id': 258, 'city': 'Goa'}, {'id': 259, 'city': 'Pune'}, {'id': 260, 'city': 'Pune'}, {'id': 261, 'city': 'Delhi'}, {'id': 262, 'city': 'Goa'}, {'id': 263, 'city': 'Pune'}, {'id': 264, 'city': 'Pune'}, {'id': 265, 'city': 'Delhi'}, {'id': 266, 'city': 'Goa'}, {'id': 267, 'city': 'Pune'}, {'id': 268, 'city': 'Pune'}, {'id': 269, 'city': 'Delhi'}, {'id': 270, 'city': 'Goa'}, {'id': 271, 'city': 'Pune'}, {'id': 272, 'city': 'Pune'}, {'id': 273, 'city': 'Delhi'}, {'id': 274, 'city': 'Goa'}, {'id': 275, 'city': 'Pune'}, {'id': 276, 'city': 'Pune'}, {'id': 277, 'city': 'Delhi'}, {'id': 278, 'city': 'Goa'}, {'id': 279, 'city': 'Pune'}, {'id': 280, 'city': 'Pune'}, {'id': 281, 'city': 'Delhi'}, {'id': 282, 'city': 'Goa'}, {'id': 283, 'city': 'Pune'}, {'id': 284, 'city': 'Pune'}, {'id': 285, 'city': 'Delhi'}, {'id': 286, 'city': 'Goa'}, {'id': 287, 'city': 'Pune'}, {'id': 288, 'city': 'Pune'}, {'id': 289, 'city': 'Delhi'}, {'id': 290, 'city': 'Goa'}, {'id': 291, 'city': 'Pune'}, {'id': 292, 'city': 'Pune'}, {'id': 293, 'city': 'Delhi'}, {'id': 294, 'city': 'Goa'}, {'id': 295, 'city': 'Pune'}, {'id': 296, 'city': 'Pune'}, {'id': 297, 'city': 'Delhi'}, {'id': 298, 'city': 'Goa'}, {'id': 299, 'city': 'Pune'}, {'id': 300, 'city': 'Pune'}, {'id': 301, 'city': 'Delhi'}, {'id': 302, 'city': 'Goa'}, {'id': 303, 'city': 'Pune'}, {'id': 304, 'city': 'Pune'}, {'id': 305, 'city': 'Delhi'}, {'id': 306, 'city': 'Goa'}, {'id': 307, 'city': 'Pune'}, {'id': 308, 'city': 'Pune'}, {'id': 309, 'city': 'Delhi'}, {'id': 310, 'city': 'Goa'}, {'id': 311, 'city': 'Pune'}, {'id': 312, 'city': 'Pune'}, {'id': 313, 'city': 'Delhi'}, {'id': 314, 'city': 'Goa'}, {'id': 315, 'city': 'Pune'}, {'id': 316, 'city': 'Pune'}, {'id': 317, 'city': 'Delhi'}, {'id': 318, 'city': 'Goa'}, {'id': 319, 'city': 'Pune'}, {'id': 320, 'city': 'Pune'}, {'id': 321, 'city': 'Delhi'}, {'id': 322, 'city': 'Goa'}, {'id': 323, 'city': 'Pune'}, {'id': 324, 'city': 'Pune'}, {'id': 325, 'city': 'Delhi'}, {'id': 326, 'city': 'Goa'}, {'id': 327, 'city': 'Pune'}, {'id': 328, 'city': 'Pune'}, {'id': 329, 'city': 'Delhi'}, {'id': 330, 'city': 'Goa'}, {'id': 331, 'city': 'Pune'}, {'id': 332, 'city': 'Pune'}, {'id': 333, 'city': 'Delhi'}, {'id': 334, 'city': 'Goa'}, {'id': 335, 'city': 'Pune'}, {'id': 336, 'city': 'Pune'}, {'id': 337, 'city': 'Delhi'}, {'id': 338, 'city': 'Goa'}, {'id': 339, 'city': 'Pune'}, {'id': 340, 'city': 'Pune'}, {'id': 341, 'city': 'Delhi'}, {'id': 342, 'city': 'Goa'}, {'id': 343, 'city': 'Pune'}, {'id': 344, 'city': 'Pune'}, {'id': 345, 'city': 'Delhi'}, {'id': 346, 'city': 'Goa'}, {'id': 347, 'city': 'Pune'}, {'id': 348, 'city': 'Pune'}, {'id': 349, 'city': 'Delhi'}, {'id': 350, 'city': 'Goa'}, {'id': 351, 'city': 'Pune'}, {'id': 352, 'city': 'Pune'}, {'id': 353, 'city': 'Delhi'}, {'id': 354, 'city': 'Goa'}, {'id': 355, 'city': 'Pune'}, {'id': 356, 'city': 'Pune'}, {'id': 357, 'city': 'Delhi'}, {'id': 358, 'city': 'Goa'}, {'id': 359, 'city': 'Pune'}, {'id': 360, 'city': 'Pune'}, {'id': 361, 'city': 'Delhi'}, {'id': 362, 'city': 'Goa'}, {'id': 363, 'city': 'Pune'}, {'id': 364, 'city': 'Pune'}, {'id': 365, 'city': 'Delhi'}, {'id': 366, 'city': 'Goa'}, {'id': 367, 'city': 'Pune'}, {'id': 368, 'city': 'Pune'}, {'id': 369, 'city': 'Delhi'}, {'id': 370, 'city': 'Goa'}, {'id': 371, 'city': 'Pune'}, {'id': 372, 'city': 'Pune'}, {'id': 373, 'city': 'Delhi'}, {'id': 374, 'city': 'Goa'}, {'id': 375, 'city': 'Pune'}, {'id': 376, 'city': 'Pune'}, {'id': 377, 'city': 'Delhi'}, {'id': 378, 'city': 'Goa'}, {'id': 379, 'city': 'Pune'}, {'id': 380, 'city': 'Pune'}, {'id': 381, 'city': 'Delhi'}, {'id': 382, 'city': 'Goa'}, {'id': 383, 'city': 'Pune'}, {'id': 384, 'city': 'Pune'}, {'id': 385, 'city': 'Delhi'}, {'id': 386, 'city': 'Goa'}, {'id': 387, 'city': 'Pune'}, {'id': 388, 'city': 'Pune'}, {'id': 389, 'city': 'Delhi'}, {'id': 390, 'city': 'Goa'}, {'id': 391, 'city': 'Pune'}, {'id': 392, 'city': 'Pune'}, {'id': 393, 'city': 'Delhi'}, {'id': 394, 'city': 'Goa'}, {'id': 395, 'city': 'Pune'}, {'id': 396, 'city': 'Pune'}, {'id': 397, 'city': 'Delhi'}, {'id': 398, 'city': 'Goa'}, {'id': 399, 'city': 'Pune'}, {'id': 400, 'city': 'Pune'}, {'id': 401, 'city': 'Delhi'}, {'id': 402, 'city': 'Goa'}, {'id': 403, 'city': 'Pune'}, {'id': 404, 'city': 'Pune'}, {'id': 405, 'city': 'Delhi'}, {'id': 406, 'city': 'Goa'}, {'id': 407, 'city': 'Pune'}, {'id': 408, 'city': 'Pune'}, {'id': 409, 'city': 'Delhi'}, {'id': 410, 'city': 'Goa'}, {'id': 411, 'city': 'Pune'}, {'id': 412, 'city': 'Pune'}, {'id': 413, 'city': 'Delhi'}, {'id': 414, 'city': 'Goa'}, {'id': 415, 'city': 'Pune'}, {'id': 416, 'city': 'Pune'}, {'id': 417, 'city': 'Delhi'}, {'id': 418, 'city': 'Goa'}, {'id': 419, 'city': 'Pune'}, {'id': 420, 'city': 'Pune'}, {'id': 421, 'city': 'Delhi'}, {'id': 422, 'city': 'Goa'}, {'id': 423, 'city': 'Pune'}, {'id': 424, 'city': 'Pune'}, {'id': 425, 'city': 'Delhi'}, {'id': 426, 'city': 'Goa'}, {'id': 427, 'city': 'Pune'}, {'id': 428, 'city': 'Pune'}, {'id': 429, 'city': 'Delhi'}, {'id': 430, 'city': 'Goa'}, {'id': 431, 'city': 'Pune'}, {'id': 432, 'city': 'Pune'}, {'id': 433, 'city': 'Delhi'}, {'id': 434, 'city': 'Goa'}, {'id': 435, 'city': 'Pune'}, {'id': 436, 'city': 'Pune'}, {'id': 437, 'city': 'Delhi'}, {'id': 438, 'city': 'Goa'}, {'id': 439, 'city': 'Pune'}, {'id': 440, 'city': 'Pune'}, {'id': 441, 'city': 'Delhi'}, {'id': 442, 'city': 'Goa'}, {'id': 443, 'city': 'Pune'}, {'id': 444, 'city': 'Pune'}, {'id': 445, 'city': 'Delhi'}, {'id': 446, 'city': 'Goa'}, {'id': 447, 'city': 'Pune'}, {'id': 448, 'city': 'Pune'}, {'id': 449, 'city': 'Delhi'}, {'id': 450, 'city': 'Goa'}, {'id': 451, 'city': 'Pune'}, {'id': 452, 'city': 'Pune'}, {'id': 453, 'city': 'Delhi'}, {'id': 454, 'city': 'Goa'}, {'id': 455, 'city': 'Pune'}, {'id': 456, 'city': 'Pune'}, {'id': 457, 'city': 'Delhi'}, {'id': 458, 'city': 'Goa'}, {'id': 459, 'city': 'Pune'}, {'id': 460, 'city': 'Pune'}, {'id': 461, 'city': 'Delhi'}, {'id': 462, 'city': 'Goa'}, {'id': 463, 'city': 'Pune'}, {'id': 464, 'city': 'Pune'}, {'id': 465, 'city': 'Delhi'}, {'id': 466, 'city': 'Goa'}, {'id': 467, 'city': 'Pune'}, {'id': 468, 'city': 'Pune'}, {'id': 469, 'city': 'Delhi'}, {'id': 470, 'city': 'Goa'}, {'id': 471, 'city': 'Pune'}, {'id': 472, 'city': 'Pune'}, {'id': 473, 'city': 'Delhi'}, {'id': 474, 'city': 'Goa'}, {'id': 475, 'city': 'Pune'}, {'id': 476, 'city': 'Pune'}, {'id': 477, 'city': 'Delhi'}, {'id': 478, 'city': 'Goa'}, {'id': 479, 'city': 'Pune'}, {'id': 480, 'city': 'Pune'}, {'id': 481, 'city': 'Delhi'}, {'id': 482, 'city': 'Goa'}, {'id': 483, 'city': 'Pune'}, {'id': 484, 'city': 'Pune'}, {'id': 485, 'city': 'Delhi'}, {'id': 486, 'city': 'Goa'}, {'id': 487, 'city': 'Pune'}, {'id': 488, 'city': 'Pune'}, {'id': 489, 'city': 'Delhi'}, {'id': 490, 'city': 'Goa'}, {'id': 491, 'city': 'Pune'}, {'id': 492, 'city': 'Pune'}, {'id': 493, 'city': 'Delhi'}, {'id': 494, 'city': 'Goa'}, {'id': 495, 'city': 'Pune'}, {'id': 496, 'city': 'Pune'}, {'id': 497, 'city': 'Delhi'}, {'id': 498, 'city': 'Goa'}, {'id': 499, 'city': 'Pune'}, {'id': 500, 'city': 'Pune'}, {'id': 501, 'city': 'Delhi'}, {'id': 502, 'city': 'Goa'}, {'id': 503, 'city': 'Pune'}, {'id': 504, 'city': 'Pune'}, {'id': 505, 'city': 'Delhi'}, {'id': 506, 'city': 'Goa'}, {'id': 507, 'city': 'Pune'}, {'id': 508, 'city': 'Pune'}, {'id': 509, 'city': 'Delhi'}, {'id': 510, 'city': 'Goa'}, {'id': 511, 'city': 'Pune'}, {'id': 512, 'city': 'Pune'}, {'id': 513, 'city': 'Delhi'}, {'id': 514, 'city': 'Goa'}, {'id': 515, 'city': 'Pune'}, {'id': 516, 'city': 'Pune'}, {'id': 517, 'city': 'Delhi'}, {'id': 518, 'city': 'Goa'}, {'id': 519, 'city': 'Pune'}, {'id': 520, 'city': 'Pune'}, {'id': 521, 'city': 'Delhi'}, {'id': 522, 'city': 'Goa'}, {'id': 523, 'city': 'Pune'}, {'id': 524, 'city': 'Pune'}, {'id': 525, 'city': 'Delhi'}, {'id': 526, 'city': 'Goa'}, {'id': 527, 'city': 'Pune'}, {'id': 528, 'city': 'Pune'}, {'id': 529, 'city': 'Delhi'}, {'id': 530, 'city': 'Goa'}, {'id': 531, 'city': 'Pune'}, {'id': 532, 'city': 'Pune'}, {'id': 533, 'city': 'Delhi'}, {'id': 534, 'city': 'Goa'}, {'id': 535, 'city': 'Pune'}, {'id': 536, 'city': 'Pune'}, {'id': 537, 'city': 'Delhi'}, {'id': 538, 'city': 'Goa'}, {'id': 539, 'city': 'Pune'}, {'id': 540, 'city': 'Pune'}, {'id': 541, 'city': 'Delhi'}, {'id': 542, 'city': 'Goa'}, {'id': 543, 'city': 'Pune'}, {'id': 544, 'city': 'Pune'}, {'id': 545, 'city': 'Delhi'}, {'id': 546, 'city': 'Goa'}, {'id': 547, 'city': 'Pune'}, {'id': 548, 'city': 'Pune'}, {'id': 549, 'city': 'Delhi'}, {'id': 550, 'city': 'Goa'}, {'id': 551, 'city': 'Pune'}, {'id': 552, 'city': 'Pune'}, {'id': 553, 'city': 'Delhi'}, {'id': 554, 'city': 'Goa'}, {'id': 555, 'city': 'Pune'}, {'id': 556, 'city': 'Pune'}, {'id': 557, 'city': 'Delhi'}, {'id': 558, 'city': 'Goa'}, {'id': 559, 'city': 'Pune'}, {'id': 560, 'city': 'Pune'}, {'id': 561, 'city': 'Delhi'}, {'id': 562, 'city': 'Goa'}, {'id': 563, 'city': 'Pune'}, {'id': 564, 'city': 'Pune'}, {'id': 565, 'city': 'Delhi'}, {'id': 566, 'city': 'Goa'}, {'id': 567, 'city': 'Pune'}, {'id': 568, 'city': 'Pune'}, {'id': 569, 'city': 'Delhi'}, {'id': 570, 'city': 'Goa'}, {'id': 571, 'city': 'Pune'}, {'id': 572, 'city': 'Pune'}, {'id': 573, 'city': 'Delhi'}, {'id': 574, 'city': 'Goa'}, {'id': 575, 'city': 'Pune'}, {'id': 576, 'city': 'Pune'}, {'id': 577, 'city': 'Delhi'}, {'id': 578, 'city': 'Goa'}, {'id': 579, 'city': 'Pune'}, {'id': 580, 'city': 'Pune'}, {'id': 581, 'city': 'Delhi'}, {'id': 582, 'city': 'Goa'}, {'id': 583, 'city': 'Pune'}, {'id': 584, 'city': 'Pune'}, {'id': 585, 'city': 'Delhi'}, {'id': 586, 'city': 'Goa'}, {'id': 587, 'city': 'Pune'}, {'id': 588, 'city': 'Pune'}, {'id': 589, 'city': 'Delhi'}, {'id': 590, 'city': 'Goa'}, {'id': 591, 'city': 'Pune'}, {'id': 592, 'city': 'Pune'}, {'id': 593, 'city': 'Delhi'}, {'id': 594, 'city': 'Goa'}, {'id': 595, 'city': 'Pune'}, {'id': 596, 'city': 'Pune'}, {'id': 597, 'city': 'Delhi'}, {'id': 598, 'city': 'Goa'}, {'id': 599, 'city': 'Pune'}, {'id': 600, 'city': 'Pune'}, {'id': 601, 'city': 'Delhi'}, {'id': 602, 'city': 'Goa'}, {'id': 603, 'city': 'Pune'}, {'id': 604, 'city': 'Pune'}, {'id': 605, 'city': 'Delhi'}, {'id': 606, 'city': 'Goa'}, {'id': 607, 'city': 'Pune'}, {'id': 608, 'city': 'Pune'}, {'id': 609, 'city': 'Delhi'}, {'id': 610, 'city': 'Goa'}, {'id': 611, 'city': 'Pune'}, {'id': 612, 'city': 'Pune'}, {'id': 613, 'city': 'Delhi'}, {'id': 614, 'city': 'Goa'}, {'id': 615, 'city': 'Pune'}, {'id': 616, 'city': 'Pune'}, {'id': 617, 'city': 'Delhi'}, {'id': 618, 'city': 'Goa'}, {'id': 619, 'city': 'Pune'}, {'id': 620, 'city': 'Pune'}, {'id': 621, 'city': 'Delhi'}, {'id': 622, 'city': 'Goa'}, {'id': 623, 'city': 'Pune'}, {'id': 624, 'city': 'Pune'}, {'id': 625, 'city': 'Delhi'}, {'id': 626, 'city': 'Goa'}, {'id': 627, 'city': 'Pune'}, {'id': 628, 'city': 'Pune'}, {'id': 629, 'city': 'Delhi'}, {'id': 630, 'city': 'Goa'}, {'id': 631, 'city': 'Pune'}, {'id': 632, 'city': 'Pune'}, {'id': 633, 'city': 'Delhi'}, {'id': 634, 'city': 'Goa'}, {'id': 635, 'city': 'Pune'}, {'id': 636, 'city': 'Pune'}, {'id': 637, 'city': 'Delhi'}, {'id': 638, 'city': 'Goa'}, {'id': 639, 'city': 'Pune'}, {'id': 640, 'city': 'Pune'}, {'id': 641, 'city': 'Delhi'}, {'id': 642, 'city': 'Goa'}, {'id': 643, 'city': 'Pune'}, {'id': 644, 'city': 'Pune'}, {'id': 645, 'city': 'Delhi'}, {'id': 646, 'city': 'Goa'}, {'id': 647, 'city': 'Pune'}, {'id': 648, 'city': 'Pune'}, {'id': 649, 'city': 'Delhi'}, {'id': 650, 'city': 'Goa'}, {'id': 651, 'city': 'Pune'}, {'id': 652, 'city': 'Pune'}, {'id': 653, 'city': 'Delhi'}, {'id': 654, 'city': 'Goa'}, {'id': 655, 'city': 'Pune'}, {'id': 656, 'city': 'Pune'}, {'id': 657, 'city': 'Delhi'}, {'id': 658, 'city': 'Goa'}, {'id': 659, 'city': 'Pune'}, {'id': 660, 'city': 'Pune'}, {'id': 661, 'city': 'Delhi'}, {'id': 662, 'city': 'Goa'}, {'id': 663, 'city': 'Pune'}, {'id': 664, 'city': 'Pune'}, {'id': 665, 'city': 'Delhi'}, {'id': 666, 'city': 'Goa'}, {'id': 667, 'city': 'Pune'}, {'id': 668, 'city': 'Pune'}, {'id': 669, 'city': 'Delhi'}, {'id': 670, 'city': 'Goa'}, {'id': 671, 'city': 'Pune'}, {'id': 672, 'city': 'Pune'}, {'id': 673, 'city': 'Delhi'}, {'id': 674, 'city': 'Goa'}, {'id': 675, 'city': 'Pune'}, {'id': 676, 'city': 'Pune'}, {'id': 677, 'city': 'Delhi'}, {'id': 678, 'city': 'Goa'}, {'id': 679, 'city': 'Pune'}, {'id': 680, 'city': 'Pune'}, {'id': 681, 'city': 'Delhi'}, {'id': 682, 'city': 'Goa'}, {'id': 683, 'city': 'Pune'}, {'id': 684, 'city': 'Pune'}, {'id': 685, 'city': 'Delhi'}, {'id': 686, 'city': 'Goa'}, {'id': 687, 'city': 'Pune'}, {'id': 688, 'city': 'Pune'}, {'id': 689, 'city': 'Delhi'}, {'id': 690, 'city': 'Goa'}, {'id': 691, 'city': 'Pune'}, {'id': 692, 'city': 'Pune'}, {'id': 693, 'city': 'Delhi'}, {'id': 694, 'city': 'Goa'}, {'id': 695, 'city': 'Pune'}, {'id': 696, 'city': 'Pune'}, {'id': 697, 'city': 'Delhi'}, {'id': 698, 'city': 'Goa'}, {'id': 699, 'city': 'Pune'}, {'id': 700, 'city': 'Pune'}, {'id': 701, 'city': 'Delhi'}, {'id': 702, 'city': 'Goa'}, {'id': 703, 'city': 'Pune'}, {'id': 704, 'city': 'Pune'}, {'id': 705, 'city': 'Delhi'}, {'id': 706, 'city': 'Goa'}, {'id': 707, 'city': 'Pune'}, {'id': 708, 'city': 'Pune'}, {'id': 709, 'city': 'Delhi'}, {'id': 710, 'city': 'Goa'}, {'id': 711, 'city': 'Pune'}, {'id': 712, 'city': 'Pune'}, {'id': 713, 'city': 'Delhi'}, {'id': 714, 'city': 'Goa'}, {'id': 715, 'city': 'Pune'}, {'id': 716, 'city': 'Pune'}, {'id': 717, 'city': 'Delhi'}, {'id': 718, 'city': 'Goa'}, {'id': 719, 'city': 'Pune'}, {'id': 720, 'city': 'Pune'}, {'id': 721, 'city': 'Delhi'}, {'id': 722, 'city': 'Goa'}, {'id': 723, 'city': 'Pune'}, {'id': 724, 'city': 'Pune'}, {'id': 725, 'city': 'Delhi'}, {'id': 726, 'city': 'Goa'}, {'id': 727, 'city': 'Pune'}, {'id': 728, 'city': 'Pune'}, {'id': 729, 'city': 'Delhi'}, {'id': 730, 'city': 'Goa'}, {'id': 731, 'city': 'Pune'}, {'id': 732, 'city': 'Pune'}, {'id': 733, 'city': 'Delhi'}, {'id': 734, 'city': 'Goa'}, {'id': 735, 'city': 'Pune'}, {'id': 736, 'city': 'Pune'}, {'id': 737, 'city': 'Delhi'}, {'id': 738, 'city': 'Goa'}, {'id': 739, 'city': 'Pune'}, {'id': 740, 'city': 'Pune'}, {'id': 741, 'city': 'Delhi'}, {'id': 742, 'city': 'Goa'}, {'id': 743, 'city': 'Pune'}, {'id': 744, 'city': 'Pune'}, {'id': 745, 'city': 'Delhi'}, {'id': 746, 'city': 'Goa'}, {'id': 747, 'city': 'Pune'}, {'id': 748, 'city': 'Pune'}, {'id': 749, 'city': 'Delhi'}, {'id': 750, 'city': 'Goa'}, {'id': 751, 'city': 'Pune'}, {'id': 752, 'city': 'Pune'}, {'id': 753, 'city': 'Delhi'}, {'id': 754, 'city': 'Goa'}, {'id': 755, 'city': 'Pune'}, {'id': 756, 'city': 'Pune'}, {'id': 757, 'city': 'Delhi'}, {'id': 758, 'city': 'Goa'}, {'id': 759, 'city': 'Pune'}, {'id': 760, 'city': 'Pune'}, {'id': 761, 'city': 'Delhi'}, {'id': 762, 'city': 'Goa'}, {'id': 763, 'city': 'Pune'}, {'id': 764, 'city': 'Pune'}, {'id': 765, 'city': 'Delhi'}, {'id': 766, 'city': 'Goa'}, {'id': 767, 'city': 'Pune'}, {'id': 768, 'city': 'Pune'}, {'id': 769, 'city': 'Delhi'}, {'id': 770, 'city': 'Goa'}, {'id': 771, 'city': 'Pune'}, {'id': 772, 'city': 'Pune'}, {'id': 773, 'city': 'Delhi'}, {'id': 774, 'city': 'Goa'}, {'id': 775, 'city': 'Pune'}, {'id': 776, 'city': 'Pune'}, {'id': 777, 'city': 'Delhi'}, {'id': 778, 'city': 'Goa'}, {'id': 779, 'city': 'Pune'}, {'id': 780, 'city': 'Pune'}, {'id': 781, 'city': 'Delhi'}, {'id': 782, 'city': 'Goa'}, {'id': 783, 'city': 'Pune'}, {'id': 784, 'city': 'Pune'}, {'id': 785, 'city': 'Delhi'}, {'id': 786, 'city': 'Goa'}, {'id': 787, 'city': 'Pune'}, {'id': 788, 'city': 'Pune'}, {'id': 789, 'city': 'Delhi'}, {'id': 790, 'city': 'Goa'}, {'id': 791, 'city': 'Pune'}, {'id': 792, 'city': 'Pune'}, {'id': 793, 'city': 'Delhi'}, {'id': 794, 'city': 'Goa'}, {'id': 795, 'city': 'Pune'}, {'id': 796, 'city': 'Pune'}, {'id': 797, 'city': 'Delhi'}, {'id': 798, 'city': 'Goa'}, {'id': 799, 'city': 'Pune'}, {'id': 800, 'city': 'Pune'}, {'id': 801, 'city': 'Delhi'}, {'id': 802, 'city': 'Goa'}, {'id': 803, 'city': 'Pune'}, {'id': 804, 'city': 'Pune'}, {'id': 805, 'city': 'Delhi'}, {'id': 806, 'city': 'Goa'}, {'id': 807, 'city': 'Pune'}, {'id': 808, 'city': 'Pune'}, {'id': 809, 'city': 'Delhi'}, {'id': 810, 'city': 'Goa'}, {'id': 811, 'city': 'Pune'}, {'id': 812, 'city': 'Pune'}, {'id': 813, 'city': 'Delhi'}, {'id': 814, 'city': 'Goa'}, {'id': 815, 'city': 'Pune'}, {'id': 816, 'city': 'Pune'}, {'id': 817, 'city': 'Delhi'}, {'id': 818, 'city': 'Goa'}, {'id': 819, 'city': 'Pune'}, {'id': 820, 'city': 'Pune'}, {'id': 821, 'city': 'Delhi'}, {'id': 822, 'city': 'Goa'}, {'id': 823, 'city': 'Pune'}, {'id': 824, 'city': 'Pune'}, {'id': 825, 'city': 'Delhi'}, {'id': 826, 'city': 'Goa'}, {'id': 827, 'city': 'Pune'}, {'id': 828, 'city': 'Pune'}, {'id': 829, 'city': 'Delhi'}, {'id': 830, 'city': 'Goa'}, {'id': 831, 'city': 'Pune'}, {'id': 832, 'city': 'Pune'}, {'id': 833, 'city': 'Delhi'}, {'id': 834, 'city': 'Goa'}, {'id': 835, 'city': 'Pune'}, {'id': 836, 'city': 'Pune'}, {'id': 837, 'city': 'Delhi'}, {'id': 838, 'city': 'Goa'}, {'id': 839, 'city': 'Pune'}, {'id': 840, 'city': 'Pune'}, {'id': 841, 'city': 'Delhi'}, {'id': 842, 'city': 'Goa'}, {'id': 843, 'city': 'Pune'}, {'id': 844, 'city': 'Pune'}, {'id': 845, 'city': 'Delhi'}, {'id': 846, 'city': 'Goa'}, {'id': 847, 'city': 'Pune'}, {'id': 848, 'city': 'Pune'}, {'id': 849, 'city': 'Delhi'}, {'id': 850, 'city': 'Goa'}, {'id': 851, 'city': 'Pune'}, {'id': 852, 'city': 'Pune'}, {'id': 853, 'city': 'Delhi'}, {'id': 854, 'city': 'Goa'}, {'id': 855, 'city': 'Pune'}, {'id': 856, 'city': 'Pune'}, {'id': 857, 'city': 'Delhi'}, {'id': 858, 'city': 'Goa'}, {'id': 859, 'city': 'Pune'}, {'id': 860, 'city': 'Pune'}, {'id': 861, 'city': 'Delhi'}, {'id': 862, 'city': 'Goa'}, {'id': 863, 'city': 'Pune'}, {'id': 864, 'city': 'Pune'}, {'id': 865, 'city': 'Delhi'}, {'id': 866, 'city': 'Goa'}, {'id': 867, 'city': 'Pune'}, {'id': 868, 'city': 'Pune'}, {'id': 869, 'city': 'Delhi'}, {'id': 870, 'city': 'Goa'}, {'id': 871, 'city': 'Pune'}, {'id': 872, 'city': 'Pune'}, {'id': 873, 'city': 'Delhi'}, {'id': 874, 'city': 'Goa'}, {'id': 875, 'city': 'Pune'}, {'id': 876, 'city': 'Pune'}, {'id': 877, 'city': 'Delhi'}, {'id': 878, 'city': 'Goa'}, {'id': 879, 'city': 'Pune'}, {'id': 880, 'city': 'Pune'}, {'id': 881, 'city': 'Delhi'}, {'id': 882, 'city': 'Goa'}, {'id': 883, 'city': 'Pune'}, {'id': 884, 'city': 'Pune'}, {'id': 885, 'city': 'Delhi'}, {'id': 886, 'city': 'Goa'}, {'id': 887, 'city': 'Pune'}, {'id': 888, 'city': 'Pune'}, {'id': 889, 'city': 'Delhi'}, {'id': 890, 'city': 'Goa'}, {'id': 891, 'city': 'Pune'}, {'id': 892, 'city': 'Pune'}, {'id': 893, 'city': 'Delhi'}, {'id': 894, 'city': 'Goa'}, {'id': 895, 'city': 'Pune'}, {'id': 896, 'city': 'Pune'}, {'id': 897, 'city': 'Delhi'}, {'id': 898, 'city': 'Goa'}, {'id': 899, 'city': 'Pune'}, {'id': 900, 'city': 'Pune'}, {'id': 901, 'city': 'Delhi'}, {'id': 902, 'city': 'Goa'}, {'id': 903, 'city': 'Pune'}, {'id': 904, 'city': 'Pune'}, {'id': 905, 'city': 'Delhi'}, {'id': 906, 'city': 'Goa'}, {'id': 907, 'city': 'Pune'}, {'id': 908, 'city': 'Pune'}, {'id': 909, 'city': 'Delhi'}, {'id': 910, 'city': 'Goa'}, {'id': 911, 'city': 'Pune'}, {'id': 912, 'city': 'Pune'}, {'id': 913, 'city': 'Delhi'}, {'id': 914, 'city': 'Goa'}, {'id': 915, 'city': 'Pune'}, {'id': 916, 'city': 'Pune'}, {'id': 917, 'city': 'Delhi'}, {'id': 918, 'city': 'Goa'}, {'id': 919, 'city': 'Pune'}, {'id': 920, 'city': 'Pune'}, {'id': 921, 'city': 'Delhi'}, {'id': 922, 'city': 'Goa'}, {'id': 923, 'city': 'Pune'}, {'id': 924, 'city': 'Pune'}, {'id': 925, 'city': 'Delhi'}, {'id': 926, 'city': 'Goa'}, {'id': 927, 'city': 'Pune'}, {'id': 928, 'city': 'Pune'}, {'id': 929, 'city': 'Delhi'}, {'id': 930, 'city': 'Goa'}, {'id': 931, 'city': 'Pune'}, {'id': 932, 'city': 'Pune'}, {'id': 933, 'city': 'Delhi'}, {'id': 934, 'city': 'Goa'}, {'id': 935, 'city': 'Pune'}, {'id': 936, 'city': 'Pune'}, {'id': 937, 'city': 'Delhi'}, {'id': 938, 'city': 'Goa'}, {'id': 939, 'city': 'Pune'}, {'id': 940, 'city': 'Pune'}, {'id': 941, 'city': 'Delhi'}, {'id': 942, 'city': 'Goa'}, {'id': 943, 'city': 'Pune'}, {'id': 944, 'city': 'Pune'}, {'id': 945, 'city': 'Delhi'}, {'id': 946, 'city': 'Goa'}, {'id': 947, 'city': 'Pune'}, {'id': 948, 'city': 'Pune'}, {'id': 949, 'city': 'Delhi'}, {'id': 950, 'city': 'Goa'}, {'id': 951, 'city': 'Pune'}, {'id': 952, 'city': 'Pune'}, {'id': 953, 'city': 'Delhi'}, {'id': 954, 'city': 'Goa'}, {'id': 955, 'city': 'Pune'}, {'id': 956, 'city': 'Pune'}, {'id': 957, 'city': 'Delhi'}, {'id': 958, 'city': 'Goa'}, {'id': 959, 'city': 'Pune'}, {'id': 960, 'city': 'Pune'}, {'id': 961, 'city': 'Delhi'}, {'id': 962, 'city': 'Goa'}, {'id': 963, 'city': 'Pune'}, {'id': 964, 'city': 'Pune'}, {'id': 965, 'city': 'Delhi'}, {'id': 966, 'city': 'Goa'}, {'id': 967, 'city': 'Pune'}, {'id': 968, 'city': 'Pune'}, {'id': 969, 'city': 'Delhi'}, {'id': 970, 'city': 'Goa'}, {'id': 971, 'city': 'Pune'}, {'id': 972, 'city': 'Pune'}, {'id': 973, 'city': 'Delhi'}, {'id': 974, 'city': 'Goa'}, {'id': 975, 'city': 'Pune'}, {'id': 976, 'city': 'Pune'}, {'id': 977, 'city': 'Delhi'}, {'id': 978, 'city': 'Goa'}, {'id': 979, 'city': 'Pune'}, {'id': 980, 'city': 'Pune'}, {'id': 981, 'city': 'Delhi'}, {'id': 982, 'city': 'Goa'}, {'id': 983, 'city': 'Pune'}, {'id': 984, 'city': 'Pune'}, {'id': 985, 'city': 'Delhi'}, {'id': 986, 'city': 'Goa'}, {'id': 987, 'city': 'Pune'}, {'id': 988, 'city': 'Pune'}, {'id': 989, 'city': 'Delhi'}, {'id': 990, 'city': 'Goa'}, {'id': 991, 'city': 'Pune'}, {'id': 992, 'city': 'Pune'}, {'id': 993, 'city': 'Delhi'}, {'id': 994, 'city': 'Goa'}, {'id': 995, 'city': 'Pune'}, {'id': 996, 'city': 'Pune'}, {'id': 997, 'city': 'Delhi'}, {'id': 998, 'city': 'Goa'}, {'id': 999, 'city': 'Pune'}], queries = [('city', 'Goa'), ('id', 7)]` → `[(1000, 260), (1000, 11)]`

In [ ]:
def index_savings(rows, queries):
    """
    Count rows examined to find all rows where field == value: a full scan reads
    every row; an index (sorted keys) costs about ⌈log₂ n⌉ steps plus one per
    match. Return (scan, indexed) for each query.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(index_savings, [
    (([{'id': 0, 'city': 'Pune'}, {'id': 1, 'city': 'Delhi'}, {'id': 2, 'city': 'Goa'}, {'id': 3, 'city': 'Pune'}, {'id': 4, 'city': 'Pune'}, {'id': 5, 'city': 'Delhi'}, {'id': 6, 'city': 'Goa'}, {'id': 7, 'city': 'Pune'}, {'id': 8, 'city': 'Pune'}, {'id': 9, 'city': 'Delhi'}, {'id': 10, 'city': 'Goa'}, {'id': 11, 'city': 'Pune'}, {'id': 12, 'city': 'Pune'}, {'id': 13, 'city': 'Delhi'}, {'id': 14, 'city': 'Goa'}, {'id': 15, 'city': 'Pune'}, {'id': 16, 'city': 'Pune'}, {'id': 17, 'city': 'Delhi'}, {'id': 18, 'city': 'Goa'}, {'id': 19, 'city': 'Pune'}, {'id': 20, 'city': 'Pune'}, {'id': 21, 'city': 'Delhi'}, {'id': 22, 'city': 'Goa'}, {'id': 23, 'city': 'Pune'}, {'id': 24, 'city': 'Pune'}, {'id': 25, 'city': 'Delhi'}, {'id': 26, 'city': 'Goa'}, {'id': 27, 'city': 'Pune'}, {'id': 28, 'city': 'Pune'}, {'id': 29, 'city': 'Delhi'}, {'id': 30, 'city': 'Goa'}, {'id': 31, 'city': 'Pune'}, {'id': 32, 'city': 'Pune'}, {'id': 33, 'city': 'Delhi'}, {'id': 34, 'city': 'Goa'}, {'id': 35, 'city': 'Pune'}, {'id': 36, 'city': 'Pune'}, {'id': 37, 'city': 'Delhi'}, {'id': 38, 'city': 'Goa'}, {'id': 39, 'city': 'Pune'}, {'id': 40, 'city': 'Pune'}, {'id': 41, 'city': 'Delhi'}, {'id': 42, 'city': 'Goa'}, {'id': 43, 'city': 'Pune'}, {'id': 44, 'city': 'Pune'}, {'id': 45, 'city': 'Delhi'}, {'id': 46, 'city': 'Goa'}, {'id': 47, 'city': 'Pune'}, {'id': 48, 'city': 'Pune'}, {'id': 49, 'city': 'Delhi'}, {'id': 50, 'city': 'Goa'}, {'id': 51, 'city': 'Pune'}, {'id': 52, 'city': 'Pune'}, {'id': 53, 'city': 'Delhi'}, {'id': 54, 'city': 'Goa'}, {'id': 55, 'city': 'Pune'}, {'id': 56, 'city': 'Pune'}, {'id': 57, 'city': 'Delhi'}, {'id': 58, 'city': 'Goa'}, {'id': 59, 'city': 'Pune'}, {'id': 60, 'city': 'Pune'}, {'id': 61, 'city': 'Delhi'}, {'id': 62, 'city': 'Goa'}, {'id': 63, 'city': 'Pune'}, {'id': 64, 'city': 'Pune'}, {'id': 65, 'city': 'Delhi'}, {'id': 66, 'city': 'Goa'}, {'id': 67, 'city': 'Pune'}, {'id': 68, 'city': 'Pune'}, {'id': 69, 'city': 'Delhi'}, {'id': 70, 'city': 'Goa'}, {'id': 71, 'city': 'Pune'}, {'id': 72, 'city': 'Pune'}, {'id': 73, 'city': 'Delhi'}, {'id': 74, 'city': 'Goa'}, {'id': 75, 'city': 'Pune'}, {'id': 76, 'city': 'Pune'}, {'id': 77, 'city': 'Delhi'}, {'id': 78, 'city': 'Goa'}, {'id': 79, 'city': 'Pune'}, {'id': 80, 'city': 'Pune'}, {'id': 81, 'city': 'Delhi'}, {'id': 82, 'city': 'Goa'}, {'id': 83, 'city': 'Pune'}, {'id': 84, 'city': 'Pune'}, {'id': 85, 'city': 'Delhi'}, {'id': 86, 'city': 'Goa'}, {'id': 87, 'city': 'Pune'}, {'id': 88, 'city': 'Pune'}, {'id': 89, 'city': 'Delhi'}, {'id': 90, 'city': 'Goa'}, {'id': 91, 'city': 'Pune'}, {'id': 92, 'city': 'Pune'}, {'id': 93, 'city': 'Delhi'}, {'id': 94, 'city': 'Goa'}, {'id': 95, 'city': 'Pune'}, {'id': 96, 'city': 'Pune'}, {'id': 97, 'city': 'Delhi'}, {'id': 98, 'city': 'Goa'}, {'id': 99, 'city': 'Pune'}, {'id': 100, 'city': 'Pune'}, {'id': 101, 'city': 'Delhi'}, {'id': 102, 'city': 'Goa'}, {'id': 103, 'city': 'Pune'}, {'id': 104, 'city': 'Pune'}, {'id': 105, 'city': 'Delhi'}, {'id': 106, 'city': 'Goa'}, {'id': 107, 'city': 'Pune'}, {'id': 108, 'city': 'Pune'}, {'id': 109, 'city': 'Delhi'}, {'id': 110, 'city': 'Goa'}, {'id': 111, 'city': 'Pune'}, {'id': 112, 'city': 'Pune'}, {'id': 113, 'city': 'Delhi'}, {'id': 114, 'city': 'Goa'}, {'id': 115, 'city': 'Pune'}, {'id': 116, 'city': 'Pune'}, {'id': 117, 'city': 'Delhi'}, {'id': 118, 'city': 'Goa'}, {'id': 119, 'city': 'Pune'}, {'id': 120, 'city': 'Pune'}, {'id': 121, 'city': 'Delhi'}, {'id': 122, 'city': 'Goa'}, {'id': 123, 'city': 'Pune'}, {'id': 124, 'city': 'Pune'}, {'id': 125, 'city': 'Delhi'}, {'id': 126, 'city': 'Goa'}, {'id': 127, 'city': 'Pune'}, {'id': 128, 'city': 'Pune'}, {'id': 129, 'city': 'Delhi'}, {'id': 130, 'city': 'Goa'}, {'id': 131, 'city': 'Pune'}, {'id': 132, 'city': 'Pune'}, {'id': 133, 'city': 'Delhi'}, {'id': 134, 'city': 'Goa'}, {'id': 135, 'city': 'Pune'}, {'id': 136, 'city': 'Pune'}, {'id': 137, 'city': 'Delhi'}, {'id': 138, 'city': 'Goa'}, {'id': 139, 'city': 'Pune'}, {'id': 140, 'city': 'Pune'}, {'id': 141, 'city': 'Delhi'}, {'id': 142, 'city': 'Goa'}, {'id': 143, 'city': 'Pune'}, {'id': 144, 'city': 'Pune'}, {'id': 145, 'city': 'Delhi'}, {'id': 146, 'city': 'Goa'}, {'id': 147, 'city': 'Pune'}, {'id': 148, 'city': 'Pune'}, {'id': 149, 'city': 'Delhi'}, {'id': 150, 'city': 'Goa'}, {'id': 151, 'city': 'Pune'}, {'id': 152, 'city': 'Pune'}, {'id': 153, 'city': 'Delhi'}, {'id': 154, 'city': 'Goa'}, {'id': 155, 'city': 'Pune'}, {'id': 156, 'city': 'Pune'}, {'id': 157, 'city': 'Delhi'}, {'id': 158, 'city': 'Goa'}, {'id': 159, 'city': 'Pune'}, {'id': 160, 'city': 'Pune'}, {'id': 161, 'city': 'Delhi'}, {'id': 162, 'city': 'Goa'}, {'id': 163, 'city': 'Pune'}, {'id': 164, 'city': 'Pune'}, {'id': 165, 'city': 'Delhi'}, {'id': 166, 'city': 'Goa'}, {'id': 167, 'city': 'Pune'}, {'id': 168, 'city': 'Pune'}, {'id': 169, 'city': 'Delhi'}, {'id': 170, 'city': 'Goa'}, {'id': 171, 'city': 'Pune'}, {'id': 172, 'city': 'Pune'}, {'id': 173, 'city': 'Delhi'}, {'id': 174, 'city': 'Goa'}, {'id': 175, 'city': 'Pune'}, {'id': 176, 'city': 'Pune'}, {'id': 177, 'city': 'Delhi'}, {'id': 178, 'city': 'Goa'}, {'id': 179, 'city': 'Pune'}, {'id': 180, 'city': 'Pune'}, {'id': 181, 'city': 'Delhi'}, {'id': 182, 'city': 'Goa'}, {'id': 183, 'city': 'Pune'}, {'id': 184, 'city': 'Pune'}, {'id': 185, 'city': 'Delhi'}, {'id': 186, 'city': 'Goa'}, {'id': 187, 'city': 'Pune'}, {'id': 188, 'city': 'Pune'}, {'id': 189, 'city': 'Delhi'}, {'id': 190, 'city': 'Goa'}, {'id': 191, 'city': 'Pune'}, {'id': 192, 'city': 'Pune'}, {'id': 193, 'city': 'Delhi'}, {'id': 194, 'city': 'Goa'}, {'id': 195, 'city': 'Pune'}, {'id': 196, 'city': 'Pune'}, {'id': 197, 'city': 'Delhi'}, {'id': 198, 'city': 'Goa'}, {'id': 199, 'city': 'Pune'}, {'id': 200, 'city': 'Pune'}, {'id': 201, 'city': 'Delhi'}, {'id': 202, 'city': 'Goa'}, {'id': 203, 'city': 'Pune'}, {'id': 204, 'city': 'Pune'}, {'id': 205, 'city': 'Delhi'}, {'id': 206, 'city': 'Goa'}, {'id': 207, 'city': 'Pune'}, {'id': 208, 'city': 'Pune'}, {'id': 209, 'city': 'Delhi'}, {'id': 210, 'city': 'Goa'}, {'id': 211, 'city': 'Pune'}, {'id': 212, 'city': 'Pune'}, {'id': 213, 'city': 'Delhi'}, {'id': 214, 'city': 'Goa'}, {'id': 215, 'city': 'Pune'}, {'id': 216, 'city': 'Pune'}, {'id': 217, 'city': 'Delhi'}, {'id': 218, 'city': 'Goa'}, {'id': 219, 'city': 'Pune'}, {'id': 220, 'city': 'Pune'}, {'id': 221, 'city': 'Delhi'}, {'id': 222, 'city': 'Goa'}, {'id': 223, 'city': 'Pune'}, {'id': 224, 'city': 'Pune'}, {'id': 225, 'city': 'Delhi'}, {'id': 226, 'city': 'Goa'}, {'id': 227, 'city': 'Pune'}, {'id': 228, 'city': 'Pune'}, {'id': 229, 'city': 'Delhi'}, {'id': 230, 'city': 'Goa'}, {'id': 231, 'city': 'Pune'}, {'id': 232, 'city': 'Pune'}, {'id': 233, 'city': 'Delhi'}, {'id': 234, 'city': 'Goa'}, {'id': 235, 'city': 'Pune'}, {'id': 236, 'city': 'Pune'}, {'id': 237, 'city': 'Delhi'}, {'id': 238, 'city': 'Goa'}, {'id': 239, 'city': 'Pune'}, {'id': 240, 'city': 'Pune'}, {'id': 241, 'city': 'Delhi'}, {'id': 242, 'city': 'Goa'}, {'id': 243, 'city': 'Pune'}, {'id': 244, 'city': 'Pune'}, {'id': 245, 'city': 'Delhi'}, {'id': 246, 'city': 'Goa'}, {'id': 247, 'city': 'Pune'}, {'id': 248, 'city': 'Pune'}, {'id': 249, 'city': 'Delhi'}, {'id': 250, 'city': 'Goa'}, {'id': 251, 'city': 'Pune'}, {'id': 252, 'city': 'Pune'}, {'id': 253, 'city': 'Delhi'}, {'id': 254, 'city': 'Goa'}, {'id': 255, 'city': 'Pune'}, {'id': 256, 'city': 'Pune'}, {'id': 257, 'city': 'Delhi'}, {'id': 258, 'city': 'Goa'}, {'id': 259, 'city': 'Pune'}, {'id': 260, 'city': 'Pune'}, {'id': 261, 'city': 'Delhi'}, {'id': 262, 'city': 'Goa'}, {'id': 263, 'city': 'Pune'}, {'id': 264, 'city': 'Pune'}, {'id': 265, 'city': 'Delhi'}, {'id': 266, 'city': 'Goa'}, {'id': 267, 'city': 'Pune'}, {'id': 268, 'city': 'Pune'}, {'id': 269, 'city': 'Delhi'}, {'id': 270, 'city': 'Goa'}, {'id': 271, 'city': 'Pune'}, {'id': 272, 'city': 'Pune'}, {'id': 273, 'city': 'Delhi'}, {'id': 274, 'city': 'Goa'}, {'id': 275, 'city': 'Pune'}, {'id': 276, 'city': 'Pune'}, {'id': 277, 'city': 'Delhi'}, {'id': 278, 'city': 'Goa'}, {'id': 279, 'city': 'Pune'}, {'id': 280, 'city': 'Pune'}, {'id': 281, 'city': 'Delhi'}, {'id': 282, 'city': 'Goa'}, {'id': 283, 'city': 'Pune'}, {'id': 284, 'city': 'Pune'}, {'id': 285, 'city': 'Delhi'}, {'id': 286, 'city': 'Goa'}, {'id': 287, 'city': 'Pune'}, {'id': 288, 'city': 'Pune'}, {'id': 289, 'city': 'Delhi'}, {'id': 290, 'city': 'Goa'}, {'id': 291, 'city': 'Pune'}, {'id': 292, 'city': 'Pune'}, {'id': 293, 'city': 'Delhi'}, {'id': 294, 'city': 'Goa'}, {'id': 295, 'city': 'Pune'}, {'id': 296, 'city': 'Pune'}, {'id': 297, 'city': 'Delhi'}, {'id': 298, 'city': 'Goa'}, {'id': 299, 'city': 'Pune'}, {'id': 300, 'city': 'Pune'}, {'id': 301, 'city': 'Delhi'}, {'id': 302, 'city': 'Goa'}, {'id': 303, 'city': 'Pune'}, {'id': 304, 'city': 'Pune'}, {'id': 305, 'city': 'Delhi'}, {'id': 306, 'city': 'Goa'}, {'id': 307, 'city': 'Pune'}, {'id': 308, 'city': 'Pune'}, {'id': 309, 'city': 'Delhi'}, {'id': 310, 'city': 'Goa'}, {'id': 311, 'city': 'Pune'}, {'id': 312, 'city': 'Pune'}, {'id': 313, 'city': 'Delhi'}, {'id': 314, 'city': 'Goa'}, {'id': 315, 'city': 'Pune'}, {'id': 316, 'city': 'Pune'}, {'id': 317, 'city': 'Delhi'}, {'id': 318, 'city': 'Goa'}, {'id': 319, 'city': 'Pune'}, {'id': 320, 'city': 'Pune'}, {'id': 321, 'city': 'Delhi'}, {'id': 322, 'city': 'Goa'}, {'id': 323, 'city': 'Pune'}, {'id': 324, 'city': 'Pune'}, {'id': 325, 'city': 'Delhi'}, {'id': 326, 'city': 'Goa'}, {'id': 327, 'city': 'Pune'}, {'id': 328, 'city': 'Pune'}, {'id': 329, 'city': 'Delhi'}, {'id': 330, 'city': 'Goa'}, {'id': 331, 'city': 'Pune'}, {'id': 332, 'city': 'Pune'}, {'id': 333, 'city': 'Delhi'}, {'id': 334, 'city': 'Goa'}, {'id': 335, 'city': 'Pune'}, {'id': 336, 'city': 'Pune'}, {'id': 337, 'city': 'Delhi'}, {'id': 338, 'city': 'Goa'}, {'id': 339, 'city': 'Pune'}, {'id': 340, 'city': 'Pune'}, {'id': 341, 'city': 'Delhi'}, {'id': 342, 'city': 'Goa'}, {'id': 343, 'city': 'Pune'}, {'id': 344, 'city': 'Pune'}, {'id': 345, 'city': 'Delhi'}, {'id': 346, 'city': 'Goa'}, {'id': 347, 'city': 'Pune'}, {'id': 348, 'city': 'Pune'}, {'id': 349, 'city': 'Delhi'}, {'id': 350, 'city': 'Goa'}, {'id': 351, 'city': 'Pune'}, {'id': 352, 'city': 'Pune'}, {'id': 353, 'city': 'Delhi'}, {'id': 354, 'city': 'Goa'}, {'id': 355, 'city': 'Pune'}, {'id': 356, 'city': 'Pune'}, {'id': 357, 'city': 'Delhi'}, {'id': 358, 'city': 'Goa'}, {'id': 359, 'city': 'Pune'}, {'id': 360, 'city': 'Pune'}, {'id': 361, 'city': 'Delhi'}, {'id': 362, 'city': 'Goa'}, {'id': 363, 'city': 'Pune'}, {'id': 364, 'city': 'Pune'}, {'id': 365, 'city': 'Delhi'}, {'id': 366, 'city': 'Goa'}, {'id': 367, 'city': 'Pune'}, {'id': 368, 'city': 'Pune'}, {'id': 369, 'city': 'Delhi'}, {'id': 370, 'city': 'Goa'}, {'id': 371, 'city': 'Pune'}, {'id': 372, 'city': 'Pune'}, {'id': 373, 'city': 'Delhi'}, {'id': 374, 'city': 'Goa'}, {'id': 375, 'city': 'Pune'}, {'id': 376, 'city': 'Pune'}, {'id': 377, 'city': 'Delhi'}, {'id': 378, 'city': 'Goa'}, {'id': 379, 'city': 'Pune'}, {'id': 380, 'city': 'Pune'}, {'id': 381, 'city': 'Delhi'}, {'id': 382, 'city': 'Goa'}, {'id': 383, 'city': 'Pune'}, {'id': 384, 'city': 'Pune'}, {'id': 385, 'city': 'Delhi'}, {'id': 386, 'city': 'Goa'}, {'id': 387, 'city': 'Pune'}, {'id': 388, 'city': 'Pune'}, {'id': 389, 'city': 'Delhi'}, {'id': 390, 'city': 'Goa'}, {'id': 391, 'city': 'Pune'}, {'id': 392, 'city': 'Pune'}, {'id': 393, 'city': 'Delhi'}, {'id': 394, 'city': 'Goa'}, {'id': 395, 'city': 'Pune'}, {'id': 396, 'city': 'Pune'}, {'id': 397, 'city': 'Delhi'}, {'id': 398, 'city': 'Goa'}, {'id': 399, 'city': 'Pune'}, {'id': 400, 'city': 'Pune'}, {'id': 401, 'city': 'Delhi'}, {'id': 402, 'city': 'Goa'}, {'id': 403, 'city': 'Pune'}, {'id': 404, 'city': 'Pune'}, {'id': 405, 'city': 'Delhi'}, {'id': 406, 'city': 'Goa'}, {'id': 407, 'city': 'Pune'}, {'id': 408, 'city': 'Pune'}, {'id': 409, 'city': 'Delhi'}, {'id': 410, 'city': 'Goa'}, {'id': 411, 'city': 'Pune'}, {'id': 412, 'city': 'Pune'}, {'id': 413, 'city': 'Delhi'}, {'id': 414, 'city': 'Goa'}, {'id': 415, 'city': 'Pune'}, {'id': 416, 'city': 'Pune'}, {'id': 417, 'city': 'Delhi'}, {'id': 418, 'city': 'Goa'}, {'id': 419, 'city': 'Pune'}, {'id': 420, 'city': 'Pune'}, {'id': 421, 'city': 'Delhi'}, {'id': 422, 'city': 'Goa'}, {'id': 423, 'city': 'Pune'}, {'id': 424, 'city': 'Pune'}, {'id': 425, 'city': 'Delhi'}, {'id': 426, 'city': 'Goa'}, {'id': 427, 'city': 'Pune'}, {'id': 428, 'city': 'Pune'}, {'id': 429, 'city': 'Delhi'}, {'id': 430, 'city': 'Goa'}, {'id': 431, 'city': 'Pune'}, {'id': 432, 'city': 'Pune'}, {'id': 433, 'city': 'Delhi'}, {'id': 434, 'city': 'Goa'}, {'id': 435, 'city': 'Pune'}, {'id': 436, 'city': 'Pune'}, {'id': 437, 'city': 'Delhi'}, {'id': 438, 'city': 'Goa'}, {'id': 439, 'city': 'Pune'}, {'id': 440, 'city': 'Pune'}, {'id': 441, 'city': 'Delhi'}, {'id': 442, 'city': 'Goa'}, {'id': 443, 'city': 'Pune'}, {'id': 444, 'city': 'Pune'}, {'id': 445, 'city': 'Delhi'}, {'id': 446, 'city': 'Goa'}, {'id': 447, 'city': 'Pune'}, {'id': 448, 'city': 'Pune'}, {'id': 449, 'city': 'Delhi'}, {'id': 450, 'city': 'Goa'}, {'id': 451, 'city': 'Pune'}, {'id': 452, 'city': 'Pune'}, {'id': 453, 'city': 'Delhi'}, {'id': 454, 'city': 'Goa'}, {'id': 455, 'city': 'Pune'}, {'id': 456, 'city': 'Pune'}, {'id': 457, 'city': 'Delhi'}, {'id': 458, 'city': 'Goa'}, {'id': 459, 'city': 'Pune'}, {'id': 460, 'city': 'Pune'}, {'id': 461, 'city': 'Delhi'}, {'id': 462, 'city': 'Goa'}, {'id': 463, 'city': 'Pune'}, {'id': 464, 'city': 'Pune'}, {'id': 465, 'city': 'Delhi'}, {'id': 466, 'city': 'Goa'}, {'id': 467, 'city': 'Pune'}, {'id': 468, 'city': 'Pune'}, {'id': 469, 'city': 'Delhi'}, {'id': 470, 'city': 'Goa'}, {'id': 471, 'city': 'Pune'}, {'id': 472, 'city': 'Pune'}, {'id': 473, 'city': 'Delhi'}, {'id': 474, 'city': 'Goa'}, {'id': 475, 'city': 'Pune'}, {'id': 476, 'city': 'Pune'}, {'id': 477, 'city': 'Delhi'}, {'id': 478, 'city': 'Goa'}, {'id': 479, 'city': 'Pune'}, {'id': 480, 'city': 'Pune'}, {'id': 481, 'city': 'Delhi'}, {'id': 482, 'city': 'Goa'}, {'id': 483, 'city': 'Pune'}, {'id': 484, 'city': 'Pune'}, {'id': 485, 'city': 'Delhi'}, {'id': 486, 'city': 'Goa'}, {'id': 487, 'city': 'Pune'}, {'id': 488, 'city': 'Pune'}, {'id': 489, 'city': 'Delhi'}, {'id': 490, 'city': 'Goa'}, {'id': 491, 'city': 'Pune'}, {'id': 492, 'city': 'Pune'}, {'id': 493, 'city': 'Delhi'}, {'id': 494, 'city': 'Goa'}, {'id': 495, 'city': 'Pune'}, {'id': 496, 'city': 'Pune'}, {'id': 497, 'city': 'Delhi'}, {'id': 498, 'city': 'Goa'}, {'id': 499, 'city': 'Pune'}, {'id': 500, 'city': 'Pune'}, {'id': 501, 'city': 'Delhi'}, {'id': 502, 'city': 'Goa'}, {'id': 503, 'city': 'Pune'}, {'id': 504, 'city': 'Pune'}, {'id': 505, 'city': 'Delhi'}, {'id': 506, 'city': 'Goa'}, {'id': 507, 'city': 'Pune'}, {'id': 508, 'city': 'Pune'}, {'id': 509, 'city': 'Delhi'}, {'id': 510, 'city': 'Goa'}, {'id': 511, 'city': 'Pune'}, {'id': 512, 'city': 'Pune'}, {'id': 513, 'city': 'Delhi'}, {'id': 514, 'city': 'Goa'}, {'id': 515, 'city': 'Pune'}, {'id': 516, 'city': 'Pune'}, {'id': 517, 'city': 'Delhi'}, {'id': 518, 'city': 'Goa'}, {'id': 519, 'city': 'Pune'}, {'id': 520, 'city': 'Pune'}, {'id': 521, 'city': 'Delhi'}, {'id': 522, 'city': 'Goa'}, {'id': 523, 'city': 'Pune'}, {'id': 524, 'city': 'Pune'}, {'id': 525, 'city': 'Delhi'}, {'id': 526, 'city': 'Goa'}, {'id': 527, 'city': 'Pune'}, {'id': 528, 'city': 'Pune'}, {'id': 529, 'city': 'Delhi'}, {'id': 530, 'city': 'Goa'}, {'id': 531, 'city': 'Pune'}, {'id': 532, 'city': 'Pune'}, {'id': 533, 'city': 'Delhi'}, {'id': 534, 'city': 'Goa'}, {'id': 535, 'city': 'Pune'}, {'id': 536, 'city': 'Pune'}, {'id': 537, 'city': 'Delhi'}, {'id': 538, 'city': 'Goa'}, {'id': 539, 'city': 'Pune'}, {'id': 540, 'city': 'Pune'}, {'id': 541, 'city': 'Delhi'}, {'id': 542, 'city': 'Goa'}, {'id': 543, 'city': 'Pune'}, {'id': 544, 'city': 'Pune'}, {'id': 545, 'city': 'Delhi'}, {'id': 546, 'city': 'Goa'}, {'id': 547, 'city': 'Pune'}, {'id': 548, 'city': 'Pune'}, {'id': 549, 'city': 'Delhi'}, {'id': 550, 'city': 'Goa'}, {'id': 551, 'city': 'Pune'}, {'id': 552, 'city': 'Pune'}, {'id': 553, 'city': 'Delhi'}, {'id': 554, 'city': 'Goa'}, {'id': 555, 'city': 'Pune'}, {'id': 556, 'city': 'Pune'}, {'id': 557, 'city': 'Delhi'}, {'id': 558, 'city': 'Goa'}, {'id': 559, 'city': 'Pune'}, {'id': 560, 'city': 'Pune'}, {'id': 561, 'city': 'Delhi'}, {'id': 562, 'city': 'Goa'}, {'id': 563, 'city': 'Pune'}, {'id': 564, 'city': 'Pune'}, {'id': 565, 'city': 'Delhi'}, {'id': 566, 'city': 'Goa'}, {'id': 567, 'city': 'Pune'}, {'id': 568, 'city': 'Pune'}, {'id': 569, 'city': 'Delhi'}, {'id': 570, 'city': 'Goa'}, {'id': 571, 'city': 'Pune'}, {'id': 572, 'city': 'Pune'}, {'id': 573, 'city': 'Delhi'}, {'id': 574, 'city': 'Goa'}, {'id': 575, 'city': 'Pune'}, {'id': 576, 'city': 'Pune'}, {'id': 577, 'city': 'Delhi'}, {'id': 578, 'city': 'Goa'}, {'id': 579, 'city': 'Pune'}, {'id': 580, 'city': 'Pune'}, {'id': 581, 'city': 'Delhi'}, {'id': 582, 'city': 'Goa'}, {'id': 583, 'city': 'Pune'}, {'id': 584, 'city': 'Pune'}, {'id': 585, 'city': 'Delhi'}, {'id': 586, 'city': 'Goa'}, {'id': 587, 'city': 'Pune'}, {'id': 588, 'city': 'Pune'}, {'id': 589, 'city': 'Delhi'}, {'id': 590, 'city': 'Goa'}, {'id': 591, 'city': 'Pune'}, {'id': 592, 'city': 'Pune'}, {'id': 593, 'city': 'Delhi'}, {'id': 594, 'city': 'Goa'}, {'id': 595, 'city': 'Pune'}, {'id': 596, 'city': 'Pune'}, {'id': 597, 'city': 'Delhi'}, {'id': 598, 'city': 'Goa'}, {'id': 599, 'city': 'Pune'}, {'id': 600, 'city': 'Pune'}, {'id': 601, 'city': 'Delhi'}, {'id': 602, 'city': 'Goa'}, {'id': 603, 'city': 'Pune'}, {'id': 604, 'city': 'Pune'}, {'id': 605, 'city': 'Delhi'}, {'id': 606, 'city': 'Goa'}, {'id': 607, 'city': 'Pune'}, {'id': 608, 'city': 'Pune'}, {'id': 609, 'city': 'Delhi'}, {'id': 610, 'city': 'Goa'}, {'id': 611, 'city': 'Pune'}, {'id': 612, 'city': 'Pune'}, {'id': 613, 'city': 'Delhi'}, {'id': 614, 'city': 'Goa'}, {'id': 615, 'city': 'Pune'}, {'id': 616, 'city': 'Pune'}, {'id': 617, 'city': 'Delhi'}, {'id': 618, 'city': 'Goa'}, {'id': 619, 'city': 'Pune'}, {'id': 620, 'city': 'Pune'}, {'id': 621, 'city': 'Delhi'}, {'id': 622, 'city': 'Goa'}, {'id': 623, 'city': 'Pune'}, {'id': 624, 'city': 'Pune'}, {'id': 625, 'city': 'Delhi'}, {'id': 626, 'city': 'Goa'}, {'id': 627, 'city': 'Pune'}, {'id': 628, 'city': 'Pune'}, {'id': 629, 'city': 'Delhi'}, {'id': 630, 'city': 'Goa'}, {'id': 631, 'city': 'Pune'}, {'id': 632, 'city': 'Pune'}, {'id': 633, 'city': 'Delhi'}, {'id': 634, 'city': 'Goa'}, {'id': 635, 'city': 'Pune'}, {'id': 636, 'city': 'Pune'}, {'id': 637, 'city': 'Delhi'}, {'id': 638, 'city': 'Goa'}, {'id': 639, 'city': 'Pune'}, {'id': 640, 'city': 'Pune'}, {'id': 641, 'city': 'Delhi'}, {'id': 642, 'city': 'Goa'}, {'id': 643, 'city': 'Pune'}, {'id': 644, 'city': 'Pune'}, {'id': 645, 'city': 'Delhi'}, {'id': 646, 'city': 'Goa'}, {'id': 647, 'city': 'Pune'}, {'id': 648, 'city': 'Pune'}, {'id': 649, 'city': 'Delhi'}, {'id': 650, 'city': 'Goa'}, {'id': 651, 'city': 'Pune'}, {'id': 652, 'city': 'Pune'}, {'id': 653, 'city': 'Delhi'}, {'id': 654, 'city': 'Goa'}, {'id': 655, 'city': 'Pune'}, {'id': 656, 'city': 'Pune'}, {'id': 657, 'city': 'Delhi'}, {'id': 658, 'city': 'Goa'}, {'id': 659, 'city': 'Pune'}, {'id': 660, 'city': 'Pune'}, {'id': 661, 'city': 'Delhi'}, {'id': 662, 'city': 'Goa'}, {'id': 663, 'city': 'Pune'}, {'id': 664, 'city': 'Pune'}, {'id': 665, 'city': 'Delhi'}, {'id': 666, 'city': 'Goa'}, {'id': 667, 'city': 'Pune'}, {'id': 668, 'city': 'Pune'}, {'id': 669, 'city': 'Delhi'}, {'id': 670, 'city': 'Goa'}, {'id': 671, 'city': 'Pune'}, {'id': 672, 'city': 'Pune'}, {'id': 673, 'city': 'Delhi'}, {'id': 674, 'city': 'Goa'}, {'id': 675, 'city': 'Pune'}, {'id': 676, 'city': 'Pune'}, {'id': 677, 'city': 'Delhi'}, {'id': 678, 'city': 'Goa'}, {'id': 679, 'city': 'Pune'}, {'id': 680, 'city': 'Pune'}, {'id': 681, 'city': 'Delhi'}, {'id': 682, 'city': 'Goa'}, {'id': 683, 'city': 'Pune'}, {'id': 684, 'city': 'Pune'}, {'id': 685, 'city': 'Delhi'}, {'id': 686, 'city': 'Goa'}, {'id': 687, 'city': 'Pune'}, {'id': 688, 'city': 'Pune'}, {'id': 689, 'city': 'Delhi'}, {'id': 690, 'city': 'Goa'}, {'id': 691, 'city': 'Pune'}, {'id': 692, 'city': 'Pune'}, {'id': 693, 'city': 'Delhi'}, {'id': 694, 'city': 'Goa'}, {'id': 695, 'city': 'Pune'}, {'id': 696, 'city': 'Pune'}, {'id': 697, 'city': 'Delhi'}, {'id': 698, 'city': 'Goa'}, {'id': 699, 'city': 'Pune'}, {'id': 700, 'city': 'Pune'}, {'id': 701, 'city': 'Delhi'}, {'id': 702, 'city': 'Goa'}, {'id': 703, 'city': 'Pune'}, {'id': 704, 'city': 'Pune'}, {'id': 705, 'city': 'Delhi'}, {'id': 706, 'city': 'Goa'}, {'id': 707, 'city': 'Pune'}, {'id': 708, 'city': 'Pune'}, {'id': 709, 'city': 'Delhi'}, {'id': 710, 'city': 'Goa'}, {'id': 711, 'city': 'Pune'}, {'id': 712, 'city': 'Pune'}, {'id': 713, 'city': 'Delhi'}, {'id': 714, 'city': 'Goa'}, {'id': 715, 'city': 'Pune'}, {'id': 716, 'city': 'Pune'}, {'id': 717, 'city': 'Delhi'}, {'id': 718, 'city': 'Goa'}, {'id': 719, 'city': 'Pune'}, {'id': 720, 'city': 'Pune'}, {'id': 721, 'city': 'Delhi'}, {'id': 722, 'city': 'Goa'}, {'id': 723, 'city': 'Pune'}, {'id': 724, 'city': 'Pune'}, {'id': 725, 'city': 'Delhi'}, {'id': 726, 'city': 'Goa'}, {'id': 727, 'city': 'Pune'}, {'id': 728, 'city': 'Pune'}, {'id': 729, 'city': 'Delhi'}, {'id': 730, 'city': 'Goa'}, {'id': 731, 'city': 'Pune'}, {'id': 732, 'city': 'Pune'}, {'id': 733, 'city': 'Delhi'}, {'id': 734, 'city': 'Goa'}, {'id': 735, 'city': 'Pune'}, {'id': 736, 'city': 'Pune'}, {'id': 737, 'city': 'Delhi'}, {'id': 738, 'city': 'Goa'}, {'id': 739, 'city': 'Pune'}, {'id': 740, 'city': 'Pune'}, {'id': 741, 'city': 'Delhi'}, {'id': 742, 'city': 'Goa'}, {'id': 743, 'city': 'Pune'}, {'id': 744, 'city': 'Pune'}, {'id': 745, 'city': 'Delhi'}, {'id': 746, 'city': 'Goa'}, {'id': 747, 'city': 'Pune'}, {'id': 748, 'city': 'Pune'}, {'id': 749, 'city': 'Delhi'}, {'id': 750, 'city': 'Goa'}, {'id': 751, 'city': 'Pune'}, {'id': 752, 'city': 'Pune'}, {'id': 753, 'city': 'Delhi'}, {'id': 754, 'city': 'Goa'}, {'id': 755, 'city': 'Pune'}, {'id': 756, 'city': 'Pune'}, {'id': 757, 'city': 'Delhi'}, {'id': 758, 'city': 'Goa'}, {'id': 759, 'city': 'Pune'}, {'id': 760, 'city': 'Pune'}, {'id': 761, 'city': 'Delhi'}, {'id': 762, 'city': 'Goa'}, {'id': 763, 'city': 'Pune'}, {'id': 764, 'city': 'Pune'}, {'id': 765, 'city': 'Delhi'}, {'id': 766, 'city': 'Goa'}, {'id': 767, 'city': 'Pune'}, {'id': 768, 'city': 'Pune'}, {'id': 769, 'city': 'Delhi'}, {'id': 770, 'city': 'Goa'}, {'id': 771, 'city': 'Pune'}, {'id': 772, 'city': 'Pune'}, {'id': 773, 'city': 'Delhi'}, {'id': 774, 'city': 'Goa'}, {'id': 775, 'city': 'Pune'}, {'id': 776, 'city': 'Pune'}, {'id': 777, 'city': 'Delhi'}, {'id': 778, 'city': 'Goa'}, {'id': 779, 'city': 'Pune'}, {'id': 780, 'city': 'Pune'}, {'id': 781, 'city': 'Delhi'}, {'id': 782, 'city': 'Goa'}, {'id': 783, 'city': 'Pune'}, {'id': 784, 'city': 'Pune'}, {'id': 785, 'city': 'Delhi'}, {'id': 786, 'city': 'Goa'}, {'id': 787, 'city': 'Pune'}, {'id': 788, 'city': 'Pune'}, {'id': 789, 'city': 'Delhi'}, {'id': 790, 'city': 'Goa'}, {'id': 791, 'city': 'Pune'}, {'id': 792, 'city': 'Pune'}, {'id': 793, 'city': 'Delhi'}, {'id': 794, 'city': 'Goa'}, {'id': 795, 'city': 'Pune'}, {'id': 796, 'city': 'Pune'}, {'id': 797, 'city': 'Delhi'}, {'id': 798, 'city': 'Goa'}, {'id': 799, 'city': 'Pune'}, {'id': 800, 'city': 'Pune'}, {'id': 801, 'city': 'Delhi'}, {'id': 802, 'city': 'Goa'}, {'id': 803, 'city': 'Pune'}, {'id': 804, 'city': 'Pune'}, {'id': 805, 'city': 'Delhi'}, {'id': 806, 'city': 'Goa'}, {'id': 807, 'city': 'Pune'}, {'id': 808, 'city': 'Pune'}, {'id': 809, 'city': 'Delhi'}, {'id': 810, 'city': 'Goa'}, {'id': 811, 'city': 'Pune'}, {'id': 812, 'city': 'Pune'}, {'id': 813, 'city': 'Delhi'}, {'id': 814, 'city': 'Goa'}, {'id': 815, 'city': 'Pune'}, {'id': 816, 'city': 'Pune'}, {'id': 817, 'city': 'Delhi'}, {'id': 818, 'city': 'Goa'}, {'id': 819, 'city': 'Pune'}, {'id': 820, 'city': 'Pune'}, {'id': 821, 'city': 'Delhi'}, {'id': 822, 'city': 'Goa'}, {'id': 823, 'city': 'Pune'}, {'id': 824, 'city': 'Pune'}, {'id': 825, 'city': 'Delhi'}, {'id': 826, 'city': 'Goa'}, {'id': 827, 'city': 'Pune'}, {'id': 828, 'city': 'Pune'}, {'id': 829, 'city': 'Delhi'}, {'id': 830, 'city': 'Goa'}, {'id': 831, 'city': 'Pune'}, {'id': 832, 'city': 'Pune'}, {'id': 833, 'city': 'Delhi'}, {'id': 834, 'city': 'Goa'}, {'id': 835, 'city': 'Pune'}, {'id': 836, 'city': 'Pune'}, {'id': 837, 'city': 'Delhi'}, {'id': 838, 'city': 'Goa'}, {'id': 839, 'city': 'Pune'}, {'id': 840, 'city': 'Pune'}, {'id': 841, 'city': 'Delhi'}, {'id': 842, 'city': 'Goa'}, {'id': 843, 'city': 'Pune'}, {'id': 844, 'city': 'Pune'}, {'id': 845, 'city': 'Delhi'}, {'id': 846, 'city': 'Goa'}, {'id': 847, 'city': 'Pune'}, {'id': 848, 'city': 'Pune'}, {'id': 849, 'city': 'Delhi'}, {'id': 850, 'city': 'Goa'}, {'id': 851, 'city': 'Pune'}, {'id': 852, 'city': 'Pune'}, {'id': 853, 'city': 'Delhi'}, {'id': 854, 'city': 'Goa'}, {'id': 855, 'city': 'Pune'}, {'id': 856, 'city': 'Pune'}, {'id': 857, 'city': 'Delhi'}, {'id': 858, 'city': 'Goa'}, {'id': 859, 'city': 'Pune'}, {'id': 860, 'city': 'Pune'}, {'id': 861, 'city': 'Delhi'}, {'id': 862, 'city': 'Goa'}, {'id': 863, 'city': 'Pune'}, {'id': 864, 'city': 'Pune'}, {'id': 865, 'city': 'Delhi'}, {'id': 866, 'city': 'Goa'}, {'id': 867, 'city': 'Pune'}, {'id': 868, 'city': 'Pune'}, {'id': 869, 'city': 'Delhi'}, {'id': 870, 'city': 'Goa'}, {'id': 871, 'city': 'Pune'}, {'id': 872, 'city': 'Pune'}, {'id': 873, 'city': 'Delhi'}, {'id': 874, 'city': 'Goa'}, {'id': 875, 'city': 'Pune'}, {'id': 876, 'city': 'Pune'}, {'id': 877, 'city': 'Delhi'}, {'id': 878, 'city': 'Goa'}, {'id': 879, 'city': 'Pune'}, {'id': 880, 'city': 'Pune'}, {'id': 881, 'city': 'Delhi'}, {'id': 882, 'city': 'Goa'}, {'id': 883, 'city': 'Pune'}, {'id': 884, 'city': 'Pune'}, {'id': 885, 'city': 'Delhi'}, {'id': 886, 'city': 'Goa'}, {'id': 887, 'city': 'Pune'}, {'id': 888, 'city': 'Pune'}, {'id': 889, 'city': 'Delhi'}, {'id': 890, 'city': 'Goa'}, {'id': 891, 'city': 'Pune'}, {'id': 892, 'city': 'Pune'}, {'id': 893, 'city': 'Delhi'}, {'id': 894, 'city': 'Goa'}, {'id': 895, 'city': 'Pune'}, {'id': 896, 'city': 'Pune'}, {'id': 897, 'city': 'Delhi'}, {'id': 898, 'city': 'Goa'}, {'id': 899, 'city': 'Pune'}, {'id': 900, 'city': 'Pune'}, {'id': 901, 'city': 'Delhi'}, {'id': 902, 'city': 'Goa'}, {'id': 903, 'city': 'Pune'}, {'id': 904, 'city': 'Pune'}, {'id': 905, 'city': 'Delhi'}, {'id': 906, 'city': 'Goa'}, {'id': 907, 'city': 'Pune'}, {'id': 908, 'city': 'Pune'}, {'id': 909, 'city': 'Delhi'}, {'id': 910, 'city': 'Goa'}, {'id': 911, 'city': 'Pune'}, {'id': 912, 'city': 'Pune'}, {'id': 913, 'city': 'Delhi'}, {'id': 914, 'city': 'Goa'}, {'id': 915, 'city': 'Pune'}, {'id': 916, 'city': 'Pune'}, {'id': 917, 'city': 'Delhi'}, {'id': 918, 'city': 'Goa'}, {'id': 919, 'city': 'Pune'}, {'id': 920, 'city': 'Pune'}, {'id': 921, 'city': 'Delhi'}, {'id': 922, 'city': 'Goa'}, {'id': 923, 'city': 'Pune'}, {'id': 924, 'city': 'Pune'}, {'id': 925, 'city': 'Delhi'}, {'id': 926, 'city': 'Goa'}, {'id': 927, 'city': 'Pune'}, {'id': 928, 'city': 'Pune'}, {'id': 929, 'city': 'Delhi'}, {'id': 930, 'city': 'Goa'}, {'id': 931, 'city': 'Pune'}, {'id': 932, 'city': 'Pune'}, {'id': 933, 'city': 'Delhi'}, {'id': 934, 'city': 'Goa'}, {'id': 935, 'city': 'Pune'}, {'id': 936, 'city': 'Pune'}, {'id': 937, 'city': 'Delhi'}, {'id': 938, 'city': 'Goa'}, {'id': 939, 'city': 'Pune'}, {'id': 940, 'city': 'Pune'}, {'id': 941, 'city': 'Delhi'}, {'id': 942, 'city': 'Goa'}, {'id': 943, 'city': 'Pune'}, {'id': 944, 'city': 'Pune'}, {'id': 945, 'city': 'Delhi'}, {'id': 946, 'city': 'Goa'}, {'id': 947, 'city': 'Pune'}, {'id': 948, 'city': 'Pune'}, {'id': 949, 'city': 'Delhi'}, {'id': 950, 'city': 'Goa'}, {'id': 951, 'city': 'Pune'}, {'id': 952, 'city': 'Pune'}, {'id': 953, 'city': 'Delhi'}, {'id': 954, 'city': 'Goa'}, {'id': 955, 'city': 'Pune'}, {'id': 956, 'city': 'Pune'}, {'id': 957, 'city': 'Delhi'}, {'id': 958, 'city': 'Goa'}, {'id': 959, 'city': 'Pune'}, {'id': 960, 'city': 'Pune'}, {'id': 961, 'city': 'Delhi'}, {'id': 962, 'city': 'Goa'}, {'id': 963, 'city': 'Pune'}, {'id': 964, 'city': 'Pune'}, {'id': 965, 'city': 'Delhi'}, {'id': 966, 'city': 'Goa'}, {'id': 967, 'city': 'Pune'}, {'id': 968, 'city': 'Pune'}, {'id': 969, 'city': 'Delhi'}, {'id': 970, 'city': 'Goa'}, {'id': 971, 'city': 'Pune'}, {'id': 972, 'city': 'Pune'}, {'id': 973, 'city': 'Delhi'}, {'id': 974, 'city': 'Goa'}, {'id': 975, 'city': 'Pune'}, {'id': 976, 'city': 'Pune'}, {'id': 977, 'city': 'Delhi'}, {'id': 978, 'city': 'Goa'}, {'id': 979, 'city': 'Pune'}, {'id': 980, 'city': 'Pune'}, {'id': 981, 'city': 'Delhi'}, {'id': 982, 'city': 'Goa'}, {'id': 983, 'city': 'Pune'}, {'id': 984, 'city': 'Pune'}, {'id': 985, 'city': 'Delhi'}, {'id': 986, 'city': 'Goa'}, {'id': 987, 'city': 'Pune'}, {'id': 988, 'city': 'Pune'}, {'id': 989, 'city': 'Delhi'}, {'id': 990, 'city': 'Goa'}, {'id': 991, 'city': 'Pune'}, {'id': 992, 'city': 'Pune'}, {'id': 993, 'city': 'Delhi'}, {'id': 994, 'city': 'Goa'}, {'id': 995, 'city': 'Pune'}, {'id': 996, 'city': 'Pune'}, {'id': 997, 'city': 'Delhi'}, {'id': 998, 'city': 'Goa'}, {'id': 999, 'city': 'Pune'}], [('city', 'Goa'), ('id', 7)]), [(1000, 260), (1000, 11)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan and count matches per query.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Build a dict index per field once.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan and count matches per query.

```python
def index_savings(rows, queries):
    out = []
    for field, value in queries:
        matches = sum(1 for r in rows if r[field] == value)
        out.append((len(rows), math.ceil(math.log2(len(rows))) + matches))
    return out
```

**Cost:** O(q · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Build a dict index per field once.

```python
def index_savings(rows, queries):
    index = defaultdict(Counter)
    for r in rows:
        for f, v in r.items():
            index[f][v] += 1
    steps = math.ceil(math.log2(len(rows)))
    return [(len(rows), steps + index[f][v]) for f, v in queries]
```

**Cost:** O(n + q) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan and count matches per query. That costs O(q · n) time and O(1) space."

"I can do better: Build a dict index per field once. That's O(n + q) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/what-an-index-saves)

<a id="q3"></a>
### Q3. Quorums: R + W &gt; N

🟢 Warm-up · *Tunable consistency*

For each `(N, W, R)`, return `(reads always see the latest write?, replica failures a write tolerates, failures a read tolerates)`.

**Examples**

- `configs = [(3, 2, 2), (3, 1, 1), (3, 3, 1), (5, 3, 3), (5, 1, 5)]` → `[(True, 1, 1), (False, 2, 2), (True, 0, 2), (True, 2, 2), (True, 4, 0)]`

In [ ]:
def quorum(configs):
    """
    For each (N, W, R), return (reads always see the latest write?, replica
    failures a write tolerates, failures a read tolerates).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(quorum, [
    (([(3, 2, 2), (3, 1, 1), (3, 3, 1), (5, 3, 3), (5, 1, 5)],), [(True, 1, 1), (False, 2, 2), (True, 0, 2), (True, 2, 2), (True, 4, 0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute each term with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute each term with if-statements.

```python
def quorum(configs):
    out = []
    for n, w, r in configs:
        strong = False
        if r + w > n:
            strong = True
        out.append((strong, n - w, n - r))
    return out
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def quorum(configs):
    return [(r + w > n, n - w, n - r) for n, w, r in configs]
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute each term with if-statements. That costs O(c) time and O(c) space."

"I can do better: A comprehension. That's O(c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/quorums-r-w-n)

<a id="q4"></a>
### Q4. Read Repair

🟢 Warm-up · *Pick the newest, fix the rest*

A quorum read got `(replica, version, value)` replies. Return the value with the highest version and the replicas (sorted) that hold an older version and need repair.

**Examples**

- `replies = [('r1', 7, 'blue'), ('r2', 9, 'green'), ('r3', 7, 'blue')]` → `('green', ['r1', 'r3'])`

In [ ]:
def read_repair(replies):
    """
    A quorum read got (replica, version, value) replies. Return the value with
    the highest version and the replicas (sorted) that hold an older version and
    need repair.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(read_repair, [
    (([('r1', 7, 'blue'), ('r2', 9, 'green'), ('r3', 7, 'blue')],), ('green', ['r1', 'r3'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Find the max version with a loop, then collect the others.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` with a key.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Find the max version with a loop, then collect the others.

```python
def read_repair(replies):
    best = replies[0]
    for rep in replies:
        if rep[1] > best[1]:
            best = rep
    stale = []
    for rep in replies:
        if rep[1] < best[1]:
            stale.append(rep[0])
    return (best[2], sorted(stale))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` with a key.

```python
def read_repair(replies):
    _, v, value = max(replies, key=lambda r: r[1])
    return (value, sorted(r for r, ver, _ in replies if ver < v))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Find the max version with a loop, then collect the others. That costs O(n) time and O(n) space."

"I can do better: `max` with a key. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/read-repair)

<a id="q5"></a>
### Q5. Hash Sharding

🟢 Warm-up · *Even spread*

Assign each key to shard `crc32(key) % n`. Return how many keys land on each shard.

**Examples**

- `keys = ['user:0', 'user:1', 'user:2', 'user:3', 'user:4', 'user:5', 'user:6', 'user:7', 'user:8', 'user:9', 'user:10', 'user:11', 'user:12', 'user:13', 'user:14', 'user:15', 'user:16', 'user:17', 'user:18', 'user:19', 'user:20', 'user:21', 'user:22', 'user:23', 'user:24', 'user:25', 'user:26', 'user:27', 'user:28', 'user:29', 'user:30', 'user:31', 'user:32', 'user:33', 'user:34', 'user:35', 'user:36', 'user:37', 'user:38', 'user:39', 'user:40', 'user:41', 'user:42', 'user:43', 'user:44', 'user:45', 'user:46', 'user:47', 'user:48', 'user:49', 'user:50', 'user:51', 'user:52', 'user:53', 'user:54', 'user:55', 'user:56', 'user:57', 'user:58', 'user:59', 'user:60', 'user:61', 'user:62', 'user:63', 'user:64', 'user:65', 'user:66', 'user:67', 'user:68', 'user:69', 'user:70', 'user:71', 'user:72', 'user:73', 'user:74', 'user:75', 'user:76', 'user:77', 'user:78', 'user:79', 'user:80', 'user:81', 'user:82', 'user:83', 'user:84', 'user:85', 'user:86', 'user:87', 'user:88', 'user:89', 'user:90', 'user:91', 'user:92', 'user:93', 'user:94', 'user:95', 'user:96', 'user:97', 'user:98', 'user:99', 'user:100', 'user:101', 'user:102', 'user:103', 'user:104', 'user:105', 'user:106', 'user:107', 'user:108', 'user:109', 'user:110', 'user:111', 'user:112', 'user:113', 'user:114', 'user:115', 'user:116', 'user:117', 'user:118', 'user:119', 'user:120', 'user:121', 'user:122', 'user:123', 'user:124', 'user:125', 'user:126', 'user:127', 'user:128', 'user:129', 'user:130', 'user:131', 'user:132', 'user:133', 'user:134', 'user:135', 'user:136', 'user:137', 'user:138', 'user:139', 'user:140', 'user:141', 'user:142', 'user:143', 'user:144', 'user:145', 'user:146', 'user:147', 'user:148', 'user:149', 'user:150', 'user:151', 'user:152', 'user:153', 'user:154', 'user:155', 'user:156', 'user:157', 'user:158', 'user:159', 'user:160', 'user:161', 'user:162', 'user:163', 'user:164', 'user:165', 'user:166', 'user:167', 'user:168', 'user:169', 'user:170', 'user:171', 'user:172', 'user:173', 'user:174', 'user:175', 'user:176', 'user:177', 'user:178', 'user:179', 'user:180', 'user:181', 'user:182', 'user:183', 'user:184', 'user:185', 'user:186', 'user:187', 'user:188', 'user:189', 'user:190', 'user:191', 'user:192', 'user:193', 'user:194', 'user:195', 'user:196', 'user:197', 'user:198', 'user:199', 'user:200', 'user:201', 'user:202', 'user:203', 'user:204', 'user:205', 'user:206', 'user:207', 'user:208', 'user:209', 'user:210', 'user:211', 'user:212', 'user:213', 'user:214', 'user:215', 'user:216', 'user:217', 'user:218', 'user:219', 'user:220', 'user:221', 'user:222', 'user:223', 'user:224', 'user:225', 'user:226', 'user:227', 'user:228', 'user:229', 'user:230', 'user:231', 'user:232', 'user:233', 'user:234', 'user:235', 'user:236', 'user:237', 'user:238', 'user:239', 'user:240', 'user:241', 'user:242', 'user:243', 'user:244', 'user:245', 'user:246', 'user:247', 'user:248', 'user:249', 'user:250', 'user:251', 'user:252', 'user:253', 'user:254', 'user:255', 'user:256', 'user:257', 'user:258', 'user:259', 'user:260', 'user:261', 'user:262', 'user:263', 'user:264', 'user:265', 'user:266', 'user:267', 'user:268', 'user:269', 'user:270', 'user:271', 'user:272', 'user:273', 'user:274', 'user:275', 'user:276', 'user:277', 'user:278', 'user:279', 'user:280', 'user:281', 'user:282', 'user:283', 'user:284', 'user:285', 'user:286', 'user:287', 'user:288', 'user:289', 'user:290', 'user:291', 'user:292', 'user:293', 'user:294', 'user:295', 'user:296', 'user:297', 'user:298', 'user:299', 'user:300', 'user:301', 'user:302', 'user:303', 'user:304', 'user:305', 'user:306', 'user:307', 'user:308', 'user:309', 'user:310', 'user:311', 'user:312', 'user:313', 'user:314', 'user:315', 'user:316', 'user:317', 'user:318', 'user:319', 'user:320', 'user:321', 'user:322', 'user:323', 'user:324', 'user:325', 'user:326', 'user:327', 'user:328', 'user:329', 'user:330', 'user:331', 'user:332', 'user:333', 'user:334', 'user:335', 'user:336', 'user:337', 'user:338', 'user:339', 'user:340', 'user:341', 'user:342', 'user:343', 'user:344', 'user:345', 'user:346', 'user:347', 'user:348', 'user:349', 'user:350', 'user:351', 'user:352', 'user:353', 'user:354', 'user:355', 'user:356', 'user:357', 'user:358', 'user:359', 'user:360', 'user:361', 'user:362', 'user:363', 'user:364', 'user:365', 'user:366', 'user:367', 'user:368', 'user:369', 'user:370', 'user:371', 'user:372', 'user:373', 'user:374', 'user:375', 'user:376', 'user:377', 'user:378', 'user:379', 'user:380', 'user:381', 'user:382', 'user:383', 'user:384', 'user:385', 'user:386', 'user:387', 'user:388', 'user:389', 'user:390', 'user:391', 'user:392', 'user:393', 'user:394', 'user:395', 'user:396', 'user:397', 'user:398', 'user:399', 'user:400', 'user:401', 'user:402', 'user:403', 'user:404', 'user:405', 'user:406', 'user:407', 'user:408', 'user:409', 'user:410', 'user:411', 'user:412', 'user:413', 'user:414', 'user:415', 'user:416', 'user:417', 'user:418', 'user:419', 'user:420', 'user:421', 'user:422', 'user:423', 'user:424', 'user:425', 'user:426', 'user:427', 'user:428', 'user:429', 'user:430', 'user:431', 'user:432', 'user:433', 'user:434', 'user:435', 'user:436', 'user:437', 'user:438', 'user:439', 'user:440', 'user:441', 'user:442', 'user:443', 'user:444', 'user:445', 'user:446', 'user:447', 'user:448', 'user:449', 'user:450', 'user:451', 'user:452', 'user:453', 'user:454', 'user:455', 'user:456', 'user:457', 'user:458', 'user:459', 'user:460', 'user:461', 'user:462', 'user:463', 'user:464', 'user:465', 'user:466', 'user:467', 'user:468', 'user:469', 'user:470', 'user:471', 'user:472', 'user:473', 'user:474', 'user:475', 'user:476', 'user:477', 'user:478', 'user:479', 'user:480', 'user:481', 'user:482', 'user:483', 'user:484', 'user:485', 'user:486', 'user:487', 'user:488', 'user:489', 'user:490', 'user:491', 'user:492', 'user:493', 'user:494', 'user:495', 'user:496', 'user:497', 'user:498', 'user:499', 'user:500', 'user:501', 'user:502', 'user:503', 'user:504', 'user:505', 'user:506', 'user:507', 'user:508', 'user:509', 'user:510', 'user:511', 'user:512', 'user:513', 'user:514', 'user:515', 'user:516', 'user:517', 'user:518', 'user:519', 'user:520', 'user:521', 'user:522', 'user:523', 'user:524', 'user:525', 'user:526', 'user:527', 'user:528', 'user:529', 'user:530', 'user:531', 'user:532', 'user:533', 'user:534', 'user:535', 'user:536', 'user:537', 'user:538', 'user:539', 'user:540', 'user:541', 'user:542', 'user:543', 'user:544', 'user:545', 'user:546', 'user:547', 'user:548', 'user:549', 'user:550', 'user:551', 'user:552', 'user:553', 'user:554', 'user:555', 'user:556', 'user:557', 'user:558', 'user:559', 'user:560', 'user:561', 'user:562', 'user:563', 'user:564', 'user:565', 'user:566', 'user:567', 'user:568', 'user:569', 'user:570', 'user:571', 'user:572', 'user:573', 'user:574', 'user:575', 'user:576', 'user:577', 'user:578', 'user:579', 'user:580', 'user:581', 'user:582', 'user:583', 'user:584', 'user:585', 'user:586', 'user:587', 'user:588', 'user:589', 'user:590', 'user:591', 'user:592', 'user:593', 'user:594', 'user:595', 'user:596', 'user:597', 'user:598', 'user:599', 'user:600', 'user:601', 'user:602', 'user:603', 'user:604', 'user:605', 'user:606', 'user:607', 'user:608', 'user:609', 'user:610', 'user:611', 'user:612', 'user:613', 'user:614', 'user:615', 'user:616', 'user:617', 'user:618', 'user:619', 'user:620', 'user:621', 'user:622', 'user:623', 'user:624', 'user:625', 'user:626', 'user:627', 'user:628', 'user:629', 'user:630', 'user:631', 'user:632', 'user:633', 'user:634', 'user:635', 'user:636', 'user:637', 'user:638', 'user:639', 'user:640', 'user:641', 'user:642', 'user:643', 'user:644', 'user:645', 'user:646', 'user:647', 'user:648', 'user:649', 'user:650', 'user:651', 'user:652', 'user:653', 'user:654', 'user:655', 'user:656', 'user:657', 'user:658', 'user:659', 'user:660', 'user:661', 'user:662', 'user:663', 'user:664', 'user:665', 'user:666', 'user:667', 'user:668', 'user:669', 'user:670', 'user:671', 'user:672', 'user:673', 'user:674', 'user:675', 'user:676', 'user:677', 'user:678', 'user:679', 'user:680', 'user:681', 'user:682', 'user:683', 'user:684', 'user:685', 'user:686', 'user:687', 'user:688', 'user:689', 'user:690', 'user:691', 'user:692', 'user:693', 'user:694', 'user:695', 'user:696', 'user:697', 'user:698', 'user:699', 'user:700', 'user:701', 'user:702', 'user:703', 'user:704', 'user:705', 'user:706', 'user:707', 'user:708', 'user:709', 'user:710', 'user:711', 'user:712', 'user:713', 'user:714', 'user:715', 'user:716', 'user:717', 'user:718', 'user:719', 'user:720', 'user:721', 'user:722', 'user:723', 'user:724', 'user:725', 'user:726', 'user:727', 'user:728', 'user:729', 'user:730', 'user:731', 'user:732', 'user:733', 'user:734', 'user:735', 'user:736', 'user:737', 'user:738', 'user:739', 'user:740', 'user:741', 'user:742', 'user:743', 'user:744', 'user:745', 'user:746', 'user:747', 'user:748', 'user:749', 'user:750', 'user:751', 'user:752', 'user:753', 'user:754', 'user:755', 'user:756', 'user:757', 'user:758', 'user:759', 'user:760', 'user:761', 'user:762', 'user:763', 'user:764', 'user:765', 'user:766', 'user:767', 'user:768', 'user:769', 'user:770', 'user:771', 'user:772', 'user:773', 'user:774', 'user:775', 'user:776', 'user:777', 'user:778', 'user:779', 'user:780', 'user:781', 'user:782', 'user:783', 'user:784', 'user:785', 'user:786', 'user:787', 'user:788', 'user:789', 'user:790', 'user:791', 'user:792', 'user:793', 'user:794', 'user:795', 'user:796', 'user:797', 'user:798', 'user:799', 'user:800', 'user:801', 'user:802', 'user:803', 'user:804', 'user:805', 'user:806', 'user:807', 'user:808', 'user:809', 'user:810', 'user:811', 'user:812', 'user:813', 'user:814', 'user:815', 'user:816', 'user:817', 'user:818', 'user:819', 'user:820', 'user:821', 'user:822', 'user:823', 'user:824', 'user:825', 'user:826', 'user:827', 'user:828', 'user:829', 'user:830', 'user:831', 'user:832', 'user:833', 'user:834', 'user:835', 'user:836', 'user:837', 'user:838', 'user:839', 'user:840', 'user:841', 'user:842', 'user:843', 'user:844', 'user:845', 'user:846', 'user:847', 'user:848', 'user:849', 'user:850', 'user:851', 'user:852', 'user:853', 'user:854', 'user:855', 'user:856', 'user:857', 'user:858', 'user:859', 'user:860', 'user:861', 'user:862', 'user:863', 'user:864', 'user:865', 'user:866', 'user:867', 'user:868', 'user:869', 'user:870', 'user:871', 'user:872', 'user:873', 'user:874', 'user:875', 'user:876', 'user:877', 'user:878', 'user:879', 'user:880', 'user:881', 'user:882', 'user:883', 'user:884', 'user:885', 'user:886', 'user:887', 'user:888', 'user:889', 'user:890', 'user:891', 'user:892', 'user:893', 'user:894', 'user:895', 'user:896', 'user:897', 'user:898', 'user:899', 'user:900', 'user:901', 'user:902', 'user:903', 'user:904', 'user:905', 'user:906', 'user:907', 'user:908', 'user:909', 'user:910', 'user:911', 'user:912', 'user:913', 'user:914', 'user:915', 'user:916', 'user:917', 'user:918', 'user:919', 'user:920', 'user:921', 'user:922', 'user:923', 'user:924', 'user:925', 'user:926', 'user:927', 'user:928', 'user:929', 'user:930', 'user:931', 'user:932', 'user:933', 'user:934', 'user:935', 'user:936', 'user:937', 'user:938', 'user:939', 'user:940', 'user:941', 'user:942', 'user:943', 'user:944', 'user:945', 'user:946', 'user:947', 'user:948', 'user:949', 'user:950', 'user:951', 'user:952', 'user:953', 'user:954', 'user:955', 'user:956', 'user:957', 'user:958', 'user:959', 'user:960', 'user:961', 'user:962', 'user:963', 'user:964', 'user:965', 'user:966', 'user:967', 'user:968', 'user:969', 'user:970', 'user:971', 'user:972', 'user:973', 'user:974', 'user:975', 'user:976', 'user:977', 'user:978', 'user:979', 'user:980', 'user:981', 'user:982', 'user:983', 'user:984', 'user:985', 'user:986', 'user:987', 'user:988', 'user:989', 'user:990', 'user:991', 'user:992', 'user:993', 'user:994', 'user:995', 'user:996', 'user:997', 'user:998', 'user:999'], n = 3` → `[330, 326, 344]`

In [ ]:
def hash_shards(keys, n):
    """
    Assign each key to shard crc32(key) % n. Return how many keys land on each
    shard.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hash_shards, [
    ((['user:0', 'user:1', 'user:2', 'user:3', 'user:4', 'user:5', 'user:6', 'user:7', 'user:8', 'user:9', 'user:10', 'user:11', 'user:12', 'user:13', 'user:14', 'user:15', 'user:16', 'user:17', 'user:18', 'user:19', 'user:20', 'user:21', 'user:22', 'user:23', 'user:24', 'user:25', 'user:26', 'user:27', 'user:28', 'user:29', 'user:30', 'user:31', 'user:32', 'user:33', 'user:34', 'user:35', 'user:36', 'user:37', 'user:38', 'user:39', 'user:40', 'user:41', 'user:42', 'user:43', 'user:44', 'user:45', 'user:46', 'user:47', 'user:48', 'user:49', 'user:50', 'user:51', 'user:52', 'user:53', 'user:54', 'user:55', 'user:56', 'user:57', 'user:58', 'user:59', 'user:60', 'user:61', 'user:62', 'user:63', 'user:64', 'user:65', 'user:66', 'user:67', 'user:68', 'user:69', 'user:70', 'user:71', 'user:72', 'user:73', 'user:74', 'user:75', 'user:76', 'user:77', 'user:78', 'user:79', 'user:80', 'user:81', 'user:82', 'user:83', 'user:84', 'user:85', 'user:86', 'user:87', 'user:88', 'user:89', 'user:90', 'user:91', 'user:92', 'user:93', 'user:94', 'user:95', 'user:96', 'user:97', 'user:98', 'user:99', 'user:100', 'user:101', 'user:102', 'user:103', 'user:104', 'user:105', 'user:106', 'user:107', 'user:108', 'user:109', 'user:110', 'user:111', 'user:112', 'user:113', 'user:114', 'user:115', 'user:116', 'user:117', 'user:118', 'user:119', 'user:120', 'user:121', 'user:122', 'user:123', 'user:124', 'user:125', 'user:126', 'user:127', 'user:128', 'user:129', 'user:130', 'user:131', 'user:132', 'user:133', 'user:134', 'user:135', 'user:136', 'user:137', 'user:138', 'user:139', 'user:140', 'user:141', 'user:142', 'user:143', 'user:144', 'user:145', 'user:146', 'user:147', 'user:148', 'user:149', 'user:150', 'user:151', 'user:152', 'user:153', 'user:154', 'user:155', 'user:156', 'user:157', 'user:158', 'user:159', 'user:160', 'user:161', 'user:162', 'user:163', 'user:164', 'user:165', 'user:166', 'user:167', 'user:168', 'user:169', 'user:170', 'user:171', 'user:172', 'user:173', 'user:174', 'user:175', 'user:176', 'user:177', 'user:178', 'user:179', 'user:180', 'user:181', 'user:182', 'user:183', 'user:184', 'user:185', 'user:186', 'user:187', 'user:188', 'user:189', 'user:190', 'user:191', 'user:192', 'user:193', 'user:194', 'user:195', 'user:196', 'user:197', 'user:198', 'user:199', 'user:200', 'user:201', 'user:202', 'user:203', 'user:204', 'user:205', 'user:206', 'user:207', 'user:208', 'user:209', 'user:210', 'user:211', 'user:212', 'user:213', 'user:214', 'user:215', 'user:216', 'user:217', 'user:218', 'user:219', 'user:220', 'user:221', 'user:222', 'user:223', 'user:224', 'user:225', 'user:226', 'user:227', 'user:228', 'user:229', 'user:230', 'user:231', 'user:232', 'user:233', 'user:234', 'user:235', 'user:236', 'user:237', 'user:238', 'user:239', 'user:240', 'user:241', 'user:242', 'user:243', 'user:244', 'user:245', 'user:246', 'user:247', 'user:248', 'user:249', 'user:250', 'user:251', 'user:252', 'user:253', 'user:254', 'user:255', 'user:256', 'user:257', 'user:258', 'user:259', 'user:260', 'user:261', 'user:262', 'user:263', 'user:264', 'user:265', 'user:266', 'user:267', 'user:268', 'user:269', 'user:270', 'user:271', 'user:272', 'user:273', 'user:274', 'user:275', 'user:276', 'user:277', 'user:278', 'user:279', 'user:280', 'user:281', 'user:282', 'user:283', 'user:284', 'user:285', 'user:286', 'user:287', 'user:288', 'user:289', 'user:290', 'user:291', 'user:292', 'user:293', 'user:294', 'user:295', 'user:296', 'user:297', 'user:298', 'user:299', 'user:300', 'user:301', 'user:302', 'user:303', 'user:304', 'user:305', 'user:306', 'user:307', 'user:308', 'user:309', 'user:310', 'user:311', 'user:312', 'user:313', 'user:314', 'user:315', 'user:316', 'user:317', 'user:318', 'user:319', 'user:320', 'user:321', 'user:322', 'user:323', 'user:324', 'user:325', 'user:326', 'user:327', 'user:328', 'user:329', 'user:330', 'user:331', 'user:332', 'user:333', 'user:334', 'user:335', 'user:336', 'user:337', 'user:338', 'user:339', 'user:340', 'user:341', 'user:342', 'user:343', 'user:344', 'user:345', 'user:346', 'user:347', 'user:348', 'user:349', 'user:350', 'user:351', 'user:352', 'user:353', 'user:354', 'user:355', 'user:356', 'user:357', 'user:358', 'user:359', 'user:360', 'user:361', 'user:362', 'user:363', 'user:364', 'user:365', 'user:366', 'user:367', 'user:368', 'user:369', 'user:370', 'user:371', 'user:372', 'user:373', 'user:374', 'user:375', 'user:376', 'user:377', 'user:378', 'user:379', 'user:380', 'user:381', 'user:382', 'user:383', 'user:384', 'user:385', 'user:386', 'user:387', 'user:388', 'user:389', 'user:390', 'user:391', 'user:392', 'user:393', 'user:394', 'user:395', 'user:396', 'user:397', 'user:398', 'user:399', 'user:400', 'user:401', 'user:402', 'user:403', 'user:404', 'user:405', 'user:406', 'user:407', 'user:408', 'user:409', 'user:410', 'user:411', 'user:412', 'user:413', 'user:414', 'user:415', 'user:416', 'user:417', 'user:418', 'user:419', 'user:420', 'user:421', 'user:422', 'user:423', 'user:424', 'user:425', 'user:426', 'user:427', 'user:428', 'user:429', 'user:430', 'user:431', 'user:432', 'user:433', 'user:434', 'user:435', 'user:436', 'user:437', 'user:438', 'user:439', 'user:440', 'user:441', 'user:442', 'user:443', 'user:444', 'user:445', 'user:446', 'user:447', 'user:448', 'user:449', 'user:450', 'user:451', 'user:452', 'user:453', 'user:454', 'user:455', 'user:456', 'user:457', 'user:458', 'user:459', 'user:460', 'user:461', 'user:462', 'user:463', 'user:464', 'user:465', 'user:466', 'user:467', 'user:468', 'user:469', 'user:470', 'user:471', 'user:472', 'user:473', 'user:474', 'user:475', 'user:476', 'user:477', 'user:478', 'user:479', 'user:480', 'user:481', 'user:482', 'user:483', 'user:484', 'user:485', 'user:486', 'user:487', 'user:488', 'user:489', 'user:490', 'user:491', 'user:492', 'user:493', 'user:494', 'user:495', 'user:496', 'user:497', 'user:498', 'user:499', 'user:500', 'user:501', 'user:502', 'user:503', 'user:504', 'user:505', 'user:506', 'user:507', 'user:508', 'user:509', 'user:510', 'user:511', 'user:512', 'user:513', 'user:514', 'user:515', 'user:516', 'user:517', 'user:518', 'user:519', 'user:520', 'user:521', 'user:522', 'user:523', 'user:524', 'user:525', 'user:526', 'user:527', 'user:528', 'user:529', 'user:530', 'user:531', 'user:532', 'user:533', 'user:534', 'user:535', 'user:536', 'user:537', 'user:538', 'user:539', 'user:540', 'user:541', 'user:542', 'user:543', 'user:544', 'user:545', 'user:546', 'user:547', 'user:548', 'user:549', 'user:550', 'user:551', 'user:552', 'user:553', 'user:554', 'user:555', 'user:556', 'user:557', 'user:558', 'user:559', 'user:560', 'user:561', 'user:562', 'user:563', 'user:564', 'user:565', 'user:566', 'user:567', 'user:568', 'user:569', 'user:570', 'user:571', 'user:572', 'user:573', 'user:574', 'user:575', 'user:576', 'user:577', 'user:578', 'user:579', 'user:580', 'user:581', 'user:582', 'user:583', 'user:584', 'user:585', 'user:586', 'user:587', 'user:588', 'user:589', 'user:590', 'user:591', 'user:592', 'user:593', 'user:594', 'user:595', 'user:596', 'user:597', 'user:598', 'user:599', 'user:600', 'user:601', 'user:602', 'user:603', 'user:604', 'user:605', 'user:606', 'user:607', 'user:608', 'user:609', 'user:610', 'user:611', 'user:612', 'user:613', 'user:614', 'user:615', 'user:616', 'user:617', 'user:618', 'user:619', 'user:620', 'user:621', 'user:622', 'user:623', 'user:624', 'user:625', 'user:626', 'user:627', 'user:628', 'user:629', 'user:630', 'user:631', 'user:632', 'user:633', 'user:634', 'user:635', 'user:636', 'user:637', 'user:638', 'user:639', 'user:640', 'user:641', 'user:642', 'user:643', 'user:644', 'user:645', 'user:646', 'user:647', 'user:648', 'user:649', 'user:650', 'user:651', 'user:652', 'user:653', 'user:654', 'user:655', 'user:656', 'user:657', 'user:658', 'user:659', 'user:660', 'user:661', 'user:662', 'user:663', 'user:664', 'user:665', 'user:666', 'user:667', 'user:668', 'user:669', 'user:670', 'user:671', 'user:672', 'user:673', 'user:674', 'user:675', 'user:676', 'user:677', 'user:678', 'user:679', 'user:680', 'user:681', 'user:682', 'user:683', 'user:684', 'user:685', 'user:686', 'user:687', 'user:688', 'user:689', 'user:690', 'user:691', 'user:692', 'user:693', 'user:694', 'user:695', 'user:696', 'user:697', 'user:698', 'user:699', 'user:700', 'user:701', 'user:702', 'user:703', 'user:704', 'user:705', 'user:706', 'user:707', 'user:708', 'user:709', 'user:710', 'user:711', 'user:712', 'user:713', 'user:714', 'user:715', 'user:716', 'user:717', 'user:718', 'user:719', 'user:720', 'user:721', 'user:722', 'user:723', 'user:724', 'user:725', 'user:726', 'user:727', 'user:728', 'user:729', 'user:730', 'user:731', 'user:732', 'user:733', 'user:734', 'user:735', 'user:736', 'user:737', 'user:738', 'user:739', 'user:740', 'user:741', 'user:742', 'user:743', 'user:744', 'user:745', 'user:746', 'user:747', 'user:748', 'user:749', 'user:750', 'user:751', 'user:752', 'user:753', 'user:754', 'user:755', 'user:756', 'user:757', 'user:758', 'user:759', 'user:760', 'user:761', 'user:762', 'user:763', 'user:764', 'user:765', 'user:766', 'user:767', 'user:768', 'user:769', 'user:770', 'user:771', 'user:772', 'user:773', 'user:774', 'user:775', 'user:776', 'user:777', 'user:778', 'user:779', 'user:780', 'user:781', 'user:782', 'user:783', 'user:784', 'user:785', 'user:786', 'user:787', 'user:788', 'user:789', 'user:790', 'user:791', 'user:792', 'user:793', 'user:794', 'user:795', 'user:796', 'user:797', 'user:798', 'user:799', 'user:800', 'user:801', 'user:802', 'user:803', 'user:804', 'user:805', 'user:806', 'user:807', 'user:808', 'user:809', 'user:810', 'user:811', 'user:812', 'user:813', 'user:814', 'user:815', 'user:816', 'user:817', 'user:818', 'user:819', 'user:820', 'user:821', 'user:822', 'user:823', 'user:824', 'user:825', 'user:826', 'user:827', 'user:828', 'user:829', 'user:830', 'user:831', 'user:832', 'user:833', 'user:834', 'user:835', 'user:836', 'user:837', 'user:838', 'user:839', 'user:840', 'user:841', 'user:842', 'user:843', 'user:844', 'user:845', 'user:846', 'user:847', 'user:848', 'user:849', 'user:850', 'user:851', 'user:852', 'user:853', 'user:854', 'user:855', 'user:856', 'user:857', 'user:858', 'user:859', 'user:860', 'user:861', 'user:862', 'user:863', 'user:864', 'user:865', 'user:866', 'user:867', 'user:868', 'user:869', 'user:870', 'user:871', 'user:872', 'user:873', 'user:874', 'user:875', 'user:876', 'user:877', 'user:878', 'user:879', 'user:880', 'user:881', 'user:882', 'user:883', 'user:884', 'user:885', 'user:886', 'user:887', 'user:888', 'user:889', 'user:890', 'user:891', 'user:892', 'user:893', 'user:894', 'user:895', 'user:896', 'user:897', 'user:898', 'user:899', 'user:900', 'user:901', 'user:902', 'user:903', 'user:904', 'user:905', 'user:906', 'user:907', 'user:908', 'user:909', 'user:910', 'user:911', 'user:912', 'user:913', 'user:914', 'user:915', 'user:916', 'user:917', 'user:918', 'user:919', 'user:920', 'user:921', 'user:922', 'user:923', 'user:924', 'user:925', 'user:926', 'user:927', 'user:928', 'user:929', 'user:930', 'user:931', 'user:932', 'user:933', 'user:934', 'user:935', 'user:936', 'user:937', 'user:938', 'user:939', 'user:940', 'user:941', 'user:942', 'user:943', 'user:944', 'user:945', 'user:946', 'user:947', 'user:948', 'user:949', 'user:950', 'user:951', 'user:952', 'user:953', 'user:954', 'user:955', 'user:956', 'user:957', 'user:958', 'user:959', 'user:960', 'user:961', 'user:962', 'user:963', 'user:964', 'user:965', 'user:966', 'user:967', 'user:968', 'user:969', 'user:970', 'user:971', 'user:972', 'user:973', 'user:974', 'user:975', 'user:976', 'user:977', 'user:978', 'user:979', 'user:980', 'user:981', 'user:982', 'user:983', 'user:984', 'user:985', 'user:986', 'user:987', 'user:988', 'user:989', 'user:990', 'user:991', 'user:992', 'user:993', 'user:994', 'user:995', 'user:996', 'user:997', 'user:998', 'user:999'], 3), [330, 326, 344]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count with a list.

```python
def hash_shards(keys, n):
    counts = [0] * n
    for k in keys:
        counts[zlib.crc32(k.encode()) % n] += 1
    return counts
```

**Cost:** O(k) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter`.

```python
def hash_shards(keys, n):
    c = Counter(zlib.crc32(k.encode()) % n for k in keys)
    return [c[i] for i in range(n)]
```

**Cost:** O(k) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with a list. That costs O(k) time and O(n) space."

"I can do better: `Counter`. That's O(k) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/hash-sharding)

<a id="q6"></a>
### Q6. Scatter-Gather Queries

🟢 Warm-up · *Queries without the shard key*

A query with a shard key goes to one shard; a query without (None) goes to every shard, waits for the slowest, then merges (`merge_ms`). Return each query's latency.

**Examples**

- `shard_ms = [12, 9, 30, 11], merge_ms = 4, queries = [0, None, 2, None, 3]` → `[12, 34, 30, 34, 11]`

In [ ]:
def scatter(shard_ms, merge_ms, queries):
    """
    A query with a shard key goes to one shard; a query without (None) goes to
    every shard, waits for the slowest, then merges (merge_ms). Return each
    query's latency.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(scatter, [
    (([12, 9, 30, 11], 4, [0, None, 2, None, 3]), [12, 34, 30, 34, 11]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If/else per query.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Precompute the scatter latency once.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If/else per query.

```python
def scatter(shard_ms, merge_ms, queries):
    out = []
    for q in queries:
        if q is None:
            out.append(max(shard_ms) + merge_ms)
        else:
            out.append(shard_ms[q])
    return out
```

**Cost:** O(q · s) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Precompute the scatter latency once.

```python
def scatter(shard_ms, merge_ms, queries):
    everywhere = max(shard_ms) + merge_ms
    return [everywhere if q is None else shard_ms[q] for q in queries]
```

**Cost:** O(q + s) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If/else per query. That costs O(q · s) time and O(1) space."

"I can do better: Precompute the scatter latency once. That's O(q + s) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/scatter-gather-queries)

<a id="q7"></a>
### Q7. Join in the Application

🟢 Warm-up · *Hash join*

After sharding, a join may happen in your code. Return `(name, total order amount)` for every user (0 when no orders), sorted by name.

**Examples**

- `users = [(1, 'Asha'), (2, 'Ravi'), (3, 'Meera')], orders = [(1, 250), (3, 99), (1, 50), (4, 10)]` → `[('Asha', 300), ('Meera', 99), ('Ravi', 0)]`

In [ ]:
def app_join(users, orders):
    """
    After sharding, a join may happen in your code. Return (name, total order
    amount) for every user (0 when no orders), sorted by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(app_join, [
    (([(1, 'Asha'), (2, 'Ravi'), (3, 'Meera')], [(1, 250), (3, 99), (1, 50), (4, 10)]), [('Asha', 300), ('Meera', 99), ('Ravi', 0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops: for each user scan all orders.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sum orders into a dict, then look up each user.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loops: for each user scan all orders.

```python
def app_join(users, orders):
    out = []
    for uid, name in users:
        total = 0
        for ouid, amount in orders:
            if ouid == uid:
                total += amount
        out.append((name, total))
    return sorted(out)
```

**Cost:** O(u · o) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sum orders into a dict, then look up each user.

```python
def app_join(users, orders):
    totals = defaultdict(int)
    for uid, amount in orders:
        totals[uid] += amount
    return sorted((name, totals[uid]) for uid, name in users)
```

**Cost:** O(u + o) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops: for each user scan all orders. That costs O(u · o) time and O(u) space."

"I can do better: Sum orders into a dict, then look up each user. That's O(u + o) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/join-in-the-application)

<a id="q8"></a>
### Q8. The N+1 Query Problem

🟢 Warm-up · *Batch your lookups*

Loading posts and then each post's author one by one costs 1 + n queries. Batching unique author ids in groups of `batch` costs 1 + ⌈unique / batch⌉. Return both counts.

**Examples**

- `author_ids = [7, 3, 7, 9, 3, 1, 7, 2, 9, 4], batch = 3` → `(11, 3)`

In [ ]:
def n_plus_one(author_ids, batch):
    """
    Loading posts and then each post's author one by one costs 1 + n queries.
    Batching unique author ids in groups of batch costs 1 + ⌈unique / batch⌉.
    Return both counts.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(n_plus_one, [
    (([7, 3, 7, 9, 3, 1, 7, 2, 9, 4], 3), (11, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count with a loop and a set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Arithmetic on `len(set(...))`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count with a loop and a set.

```python
def n_plus_one(author_ids, batch):
    naive = 1
    for _ in author_ids:
        naive += 1
    unique = []
    for a in author_ids:
        if a not in unique:
            unique.append(a)
    batched = 1
    remaining = len(unique)
    while remaining > 0:
        batched += 1
        remaining -= batch
    return (naive, batched)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Arithmetic on `len(set(...))`.

```python
def n_plus_one(author_ids, batch):
    return (1 + len(author_ids), 1 + math.ceil(len(set(author_ids)) / batch))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count with a loop and a set. That costs O(n) time and O(n) space."

"I can do better: Arithmetic on `len(set(...))`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/the-n-1-query-problem)

<a id="q9"></a>
### Q9. Optimistic Locking

🟢 Warm-up · *Compare the version, then write*

A row has `value` and `version`. Each update `(client, expected_version, new_value)` succeeds only if the version still matches, then bumps it. Return `(final value, final version, rejected clients)`.

**Examples**

- `value = 'draft', version = 1, updates = [('alice', 1, 'v-alice'), ('bob', 1, 'v-bob'), ('bob', 2, 'v-bob2'), ('carol', 2, 'v-carol')]` → `('v-bob2', 3, ['bob', 'carol'])`

In [ ]:
def optimistic(value, version, updates):
    """
    A row has value and version. Each update (client, expected_version,
    new_value) succeeds only if the version still matches, then bumps it. Return
    (final value, final version, rejected clients).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(optimistic, [
    (('draft', 1, [('alice', 1, 'v-alice'), ('bob', 1, 'v-bob'), ('bob', 2, 'v-bob2'), ('carol', 2, 'v-carol')]), ('v-bob2', 3, ['bob', 'carol'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same loop with tuple updates.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with if-statements.

```python
def optimistic(value, version, updates):
    rejected = []
    for client, expected, new in updates:
        if expected == version:
            value = new
            version = version + 1
        else:
            rejected.append(client)
    return (value, version, rejected)
```

**Cost:** O(u) time, O(u) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same loop with tuple updates.

```python
def optimistic(value, version, updates):
    rejected = []
    for client, expected, new in updates:
        if expected != version:
            rejected.append(client)
            continue
        value, version = new, version + 1
    return (value, version, rejected)
```

**Cost:** O(u) time, O(u) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with if-statements. That costs O(u) time and O(u) space."

"I can do better: Same loop with tuple updates. That's O(u) time and O(u) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/optimistic-locking)

<a id="q10"></a>
### Q10. How Many Database Connections?

🟢 Warm-up · *Little's law*

Connections in use = QPS × query time. Return `(connections needed, pool size per app instance, fits under the database's max connections?)`.

**Examples**

- `qps_ = 4000, query_ms = 15, instances = 12, db_max = 100` → `(60, 5, True)`
- `qps_ = 20000, query_ms = 25, instances = 40, db_max = 300` → `(500, 13, False)`

In [ ]:
def pool_size(qps_, query_ms, instances, db_max):
    """
    Connections in use = QPS × query time. Return (connections needed, pool size
    per app instance, fits under the database's max connections?).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pool_size, [
    ((4000, 15, 12, 100), (60, 5, True)),
    ((20000, 25, 40, 300), (500, 13, False)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply and ceiling-divide with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.ceil`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Multiply and ceiling-divide with loops.

```python
def pool_size(qps_, query_ms, instances, db_max):
    needed = qps_ * query_ms / 1000
    if needed != int(needed):
        needed = int(needed) + 1
    needed = int(needed)
    per = needed // instances
    if per * instances < needed:
        per += 1
    return (needed, per, needed <= db_max)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.ceil`.

```python
def pool_size(qps_, query_ms, instances, db_max):
    needed = math.ceil(qps_ * query_ms / 1000)
    return (needed, math.ceil(needed / instances), needed <= db_max)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply and ceiling-divide with loops. That costs O(1) time and O(1) space."

"I can do better: `math.ceil`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/how-many-database-connections)

<a id="q11"></a>
### Q11. How Many Read Replicas?

🟢 Warm-up · *Replicas also apply every write*

Every node can serve `node_qps`; each replica spends `write_qps` of that applying replicated writes. Reads go to replicas only. Return `(replicas needed, primary utilisation %)`, rounded to 1.

**Examples**

- `read_qps = 50000, write_qps = 2000, node_qps = 10000` → `(7, 20.0)`
- `read_qps = 9000, write_qps = 6000, node_qps = 8000` → `(5, 75.0)`

In [ ]:
def read_replicas(read_qps, write_qps, node_qps):
    """
    Every node can serve node_qps; each replica spends write_qps of that
    applying replicated writes. Reads go to replicas only. Return (replicas
    needed, primary utilisation %), rounded to 1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(read_replicas, [
    ((50000, 2000, 10000), (7, 20.0)),
    ((9000, 6000, 8000), (5, 75.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add replicas until capacity covers the reads.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add replicas until capacity covers the reads.

```python
def read_replicas(read_qps, write_qps, node_qps):
    replicas = 0
    while replicas * (node_qps - write_qps) < read_qps:
        replicas += 1
    return (replicas, round(write_qps / node_qps * 100, 1))
```

**Cost:** O(r) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Ceiling division.

```python
def read_replicas(read_qps, write_qps, node_qps):
    return (math.ceil(read_qps / (node_qps - write_qps)), round(write_qps / node_qps * 100, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add replicas until capacity covers the reads. That costs O(r) time and O(1) space."

"I can do better: Ceiling division. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/how-many-read-replicas)

<a id="q12"></a>
### Q12. The Leftmost-Prefix Rule

🟡 Interview standard · *Composite indexes*

An index on columns `cols` (in order) can be used for a query only from the left: stop at the first column the query doesn't filter on, and after the first column filtered by a range. Each query maps columns to 'eq' or 'range'. Return how many index columns each query can use.

**Examples**

- `cols = ['country', 'city', 'age'], queries = [{'country': 'eq', 'city': 'eq'}, {'city': 'eq'}, {'country': 'eq', 'age': 'range'}, {'country': 'range', 'city': 'eq'}, {'country': 'eq', 'city': 'eq', 'age': 'range'}]` → `[2, 0, 1, 1, 3]`

In [ ]:
def usable_columns(cols, queries):
    """
    An index on columns cols (in order) can be used for a query only from the
    left: stop at the first column the query doesn't filter on, and after the
    first column filtered by a range. Each query maps columns to 'eq' or
    'range'. Return how many index columns each query can use.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(usable_columns, [
    ((['country', 'city', 'age'], [{'country': 'eq', 'city': 'eq'}, {'city': 'eq'}, {'country': 'eq', 'age': 'range'}, {'country': 'range', 'city': 'eq'}, {'country': 'eq', 'city': 'eq', 'age': 'range'}]), [2, 0, 1, 1, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the index columns with a counter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same walk with an early break.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the index columns with a counter.

```python
def usable_columns(cols, queries):
    out = []
    for q in queries:
        used = 0
        for c in cols:
            if c not in q:
                break
            used += 1
            if q[c] == "range":
                break
        out.append(used)
    return out
```

**Cost:** O(q · c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same walk with an early break.

```python
def usable_columns(cols, queries):
    def use(q):
        n = 0
        for c in cols:
            if c not in q:
                return n
            n += 1
            if q[c] == "range":
                return n
        return n
    return [use(q) for q in queries]
```

**Cost:** O(q · c) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the index columns with a counter. That costs O(q · c) time and O(1) space."

"I can do better: Same walk with an early break. That's O(q · c) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `cols = ['a', 'b'], queries = [{'a': 'eq', 'b': 'eq'}, {'b': 'eq'}]`

1. use() returns 2
2. use() returns 0
3. usable\_columns() returns [2, 0]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/the-leftmost-prefix-rule)

<a id="q13"></a>
### Q13. Read From an LSM Tree

🟡 Interview standard · *Memtable, then SSTables*

An LSM store checks the in-memory `memtable`, then the on-disk `sstables` from newest to oldest; the first hit wins and a value of None is a tombstone (deleted). Return the value for each key (None if deleted or absent) and how many tables were checked in total.

**Examples**

- `memtable = {'k3': 30}, sstables = [{'k1': 11, 'k2': None}, {'k1': 10, 'k2': 20, 'k4': 40}], keys = ['k1', 'k2', 'k3', 'k4', 'k5']` → `([11, None, 30, 40, None], 11)`

In [ ]:
def lsm_get(memtable, sstables, keys):
    """
    An LSM store checks the in-memory memtable, then the on-disk sstables from
    newest to oldest; the first hit wins and a value of None is a tombstone
    (deleted). Return the value for each key (None if deleted or absent) and how
    many tables were checked in total.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lsm_get, [
    (({'k3': 30}, [{'k1': 11, 'k2': None}, {'k1': 10, 'k2': 20, 'k4': 40}], ['k1', 'k2', 'k3', 'k4', 'k5']), ([11, None, 30, 40, None], 11)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop through the tables for each key, counting.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, with a generator over the table list.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop through the tables for each key, counting.

```python
def lsm_get(memtable, sstables, keys):
    values, checked = [], 0
    for k in keys:
        found = None
        for table in [memtable] + sstables:
            checked += 1
            if k in table:
                found = table[k]
                break
        values.append(found)
    return (values, checked)
```

**Cost:** O(k · t) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, with a generator over the table list.

```python
def lsm_get(memtable, sstables, keys):
    tables, values, checked = [memtable, *sstables], [], 0
    for k in keys:
        i = next((i for i, t in enumerate(tables) if k in t), None)
        checked += len(tables) if i is None else i + 1
        values.append(None if i is None else tables[i][k])
    return (values, checked)
```

**Cost:** O(k · t) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop through the tables for each key, counting. That costs O(k · t) time and O(1) space."

"I can do better: Same, with a generator over the table list. That's O(k · t) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `memtable = {'a': 1}, sstables = [{'b': 2}], keys = ['a', 'b']`

1. GeneratorExit raised: 
2. GeneratorExit raised: 
3. lsm\_get() returns ([1, 2], 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/read-from-an-lsm-tree)

<a id="q14"></a>
### Q14. Compact the SSTables

🟡 Interview standard · *Merge and drop tombstones*

Compaction merges SSTables (given oldest first) into one: the newest value of each key wins, and tombstones (None) are dropped. Return the sorted `(key, value)` pairs.

**Examples**

- `sstables = [{'a': 1, 'b': 2, 'c': 3}, {'b': 20, 'd': 4}, {'a': None, 'e': 5}]` → `[('b', 20), ('c', 3), ('d', 4), ('e', 5)]`

In [ ]:
def compact(sstables):
    """
    Compaction merges SSTables (given oldest first) into one: the newest value
    of each key wins, and tombstones (None) are dropped. Return the sorted (key,
    value) pairs.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compact, [
    (([{'a': 1, 'b': 2, 'c': 3}, {'b': 20, 'd': 4}, {'a': None, 'e': 5}],), [('b', 20), ('c', 3), ('d', 4), ('e', 5)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Update a dict table by table.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ChainMap` over the tables newest first.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Update a dict table by table.

```python
def compact(sstables):
    merged = {}
    for t in sstables:
        for k, v in t.items():
            merged[k] = v
    return sorted((k, v) for k, v in merged.items() if v is not None)
```

**Cost:** O(total) time, O(keys) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `ChainMap` over the tables newest first.

```python
def compact(sstables):
    view = ChainMap(*reversed(sstables))
    return sorted((k, view[k]) for k in view if view[k] is not None)
```

**Cost:** O(total) time, O(keys) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Update a dict table by table. That costs O(total) time and O(keys) space."

"I can do better: `ChainMap` over the tables newest first. That's O(total) time and O(keys) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/compact-the-sstables)

<a id="q15"></a>
### Q15. Reading From a Lagging Replica

🟡 Interview standard · *Async replication*

Writes `(time, key, value)` reach the replica `lag` seconds later. For each read `(time, key)` served by the replica, return the value it sees (starting from `initial`) and how many reads were stale compared with the primary.

**Examples**

- `initial = {'bal': 100}, writes = [(1, 'bal', 80), (5, 'bal', 50)], lag = 3, reads = [(2, 'bal'), (4, 'bal'), (6, 'bal'), (9, 'bal')]` → `([100, 80, 80, 50], 2)`

In [ ]:
def replica_reads(initial, writes, lag, reads):
    """
    Writes (time, key, value) reach the replica lag seconds later. For each read
    (time, key) served by the replica, return the value it sees (starting from
    initial) and how many reads were stale compared with the primary.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(replica_reads, [
    (({'bal': 100}, [(1, 'bal', 80), (5, 'bal', 50)], 3, [(2, 'bal'), (4, 'bal'), (6, 'bal'), (9, 'bal')]), ([100, 80, 80, 50], 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Replay applicable writes for each read.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, as the last visible write per key.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Replay applicable writes for each read.

```python
def replica_reads(initial, writes, lag, reads):
    seen, stale = [], 0
    for t, k in reads:
        replica, primary = dict(initial), dict(initial)
        for wt, wk, wv in writes:
            if wt <= t:
                primary[wk] = wv
            if wt + lag <= t:
                replica[wk] = wv
        seen.append(replica[k])
        if replica[k] != primary[k]:
            stale += 1
    return (seen, stale)
```

**Cost:** O(r · w) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, as the last visible write per key.

```python
def replica_reads(initial, writes, lag, reads):
    def value(k, t):
        return next((v for wt, wk, v in reversed(writes) if wk == k and wt <= t), initial[k])
    seen = [value(k, t - lag) for t, k in reads]
    return (seen, sum(s != value(k, t) for s, (t, k) in zip(seen, reads)))
```

**Cost:** O(r · w) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Replay applicable writes for each read. That costs O(r · w) time and O(k) space."

"I can do better: Same, as the last visible write per key. That's O(r · w) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `initial = {'x': 0}, writes = [(1, 'x', 1)], lag = 2, reads = [(2, 'x'), (3, 'x')]`

1. value() returns 0
2. GeneratorExit raised: 
3. value() returns 1
4. GeneratorExit raised: 
5. value() returns 1
6. GeneratorExit raised: 
7. value() returns 1
8. replica\_reads() returns ([0, 1], 1)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/reading-from-a-lagging-replica)

<a id="q16"></a>
### Q16. Range Sharding and the Hot Shard

🟡 Interview standard · *Sequential keys pile up*

Shards own key ranges split at `bounds` (shard i holds keys below bounds[i]; the last shard holds the rest). Return keys per shard. Try it with timestamps: every new write lands on the last shard.

**Examples**

- `bounds = [100, 200, 300], keys = [5, 150, 210, 250, 299, 320, 330, 340, 350, 360]` → `[1, 1, 3, 5]`

In [ ]:
import bisect

def range_shards(bounds, keys):
    """
    Shards own key ranges split at bounds (shard i holds keys below bounds[i];
    the last shard holds the rest). Return keys per shard. Try it with
    timestamps: every new write lands on the last shard.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(range_shards, [
    (([100, 200, 300], [5, 150, 210, 250, 299, 320, 330, 340, 350, 360]), [1, 1, 3, 5]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare each key with the bounds in order.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`bisect_right`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare each key with the bounds in order.

```python
def range_shards(bounds, keys):
    counts = [0] * (len(bounds) + 1)
    for k in keys:
        i = 0
        while i < len(bounds) and k >= bounds[i]:
            i += 1
        counts[i] += 1
    return counts
```

**Cost:** O(k · b) time, O(b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `bisect_right`.

```python
def range_shards(bounds, keys):
    c = Counter(bisect.bisect_right(bounds, k) for k in keys)
    return [c[i] for i in range(len(bounds) + 1)]
```

**Cost:** O(k log b) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare each key with the bounds in order. That costs O(k · b) time and O(b) space."

"I can do better: `bisect_right`. That's O(k log b) time and O(b) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/range-sharding-and-the-hot-shard)

<a id="q17"></a>
### Q17. Choose a Shard Key

🟡 Interview standard · *Avoid skew*

Hash-shard `records` on each candidate field into `n` shards. Return the largest shard's share (%, rounded to 1) for each candidate, and the candidate with the smallest maximum.

**Examples**

- `records = [{'user': 'u0', 'country': 'IN', 'day': 'd0'}, {'user': 'u1', 'country': 'IN', 'day': 'd1'}, {'user': 'u2', 'country': 'IN', 'day': 'd2'}, {'user': 'u3', 'country': 'IN', 'day': 'd0'}, {'user': 'u4', 'country': 'IN', 'day': 'd1'}, {'user': 'u5', 'country': 'IN', 'day': 'd2'}, {'user': 'u6', 'country': 'IN', 'day': 'd0'}, {'user': 'u7', 'country': 'IN', 'day': 'd1'}, {'user': 'u8', 'country': 'US', 'day': 'd2'}, {'user': 'u9', 'country': 'US', 'day': 'd0'}, {'user': 'u10', 'country': 'IN', 'day': 'd1'}, {'user': 'u11', 'country': 'IN', 'day': 'd2'}, {'user': 'u12', 'country': 'IN', 'day': 'd0'}, {'user': 'u13', 'country': 'IN', 'day': 'd1'}, {'user': 'u14', 'country': 'IN', 'day': 'd2'}, {'user': 'u15', 'country': 'IN', 'day': 'd0'}, {'user': 'u16', 'country': 'IN', 'day': 'd1'}, {'user': 'u17', 'country': 'IN', 'day': 'd2'}, {'user': 'u18', 'country': 'US', 'day': 'd0'}, {'user': 'u19', 'country': 'US', 'day': 'd1'}, {'user': 'u20', 'country': 'IN', 'day': 'd2'}, {'user': 'u21', 'country': 'IN', 'day': 'd0'}, {'user': 'u22', 'country': 'IN', 'day': 'd1'}, {'user': 'u23', 'country': 'IN', 'day': 'd2'}, {'user': 'u24', 'country': 'IN', 'day': 'd0'}, {'user': 'u25', 'country': 'IN', 'day': 'd1'}, {'user': 'u26', 'country': 'IN', 'day': 'd2'}, {'user': 'u27', 'country': 'IN', 'day': 'd0'}, {'user': 'u28', 'country': 'US', 'day': 'd1'}, {'user': 'u29', 'country': 'US', 'day': 'd2'}, {'user': 'u30', 'country': 'IN', 'day': 'd0'}, {'user': 'u31', 'country': 'IN', 'day': 'd1'}, {'user': 'u32', 'country': 'IN', 'day': 'd2'}, {'user': 'u33', 'country': 'IN', 'day': 'd0'}, {'user': 'u34', 'country': 'IN', 'day': 'd1'}, {'user': 'u35', 'country': 'IN', 'day': 'd2'}, {'user': 'u36', 'country': 'IN', 'day': 'd0'}, {'user': 'u37', 'country': 'IN', 'day': 'd1'}, {'user': 'u38', 'country': 'US', 'day': 'd2'}, {'user': 'u39', 'country': 'US', 'day': 'd0'}, {'user': 'u40', 'country': 'IN', 'day': 'd1'}, {'user': 'u41', 'country': 'IN', 'day': 'd2'}, {'user': 'u42', 'country': 'IN', 'day': 'd0'}, {'user': 'u43', 'country': 'IN', 'day': 'd1'}, {'user': 'u44', 'country': 'IN', 'day': 'd2'}, {'user': 'u45', 'country': 'IN', 'day': 'd0'}, {'user': 'u46', 'country': 'IN', 'day': 'd1'}, {'user': 'u47', 'country': 'IN', 'day': 'd2'}, {'user': 'u48', 'country': 'US', 'day': 'd0'}, {'user': 'u49', 'country': 'US', 'day': 'd1'}, {'user': 'u50', 'country': 'IN', 'day': 'd2'}, {'user': 'u51', 'country': 'IN', 'day': 'd0'}, {'user': 'u52', 'country': 'IN', 'day': 'd1'}, {'user': 'u53', 'country': 'IN', 'day': 'd2'}, {'user': 'u54', 'country': 'IN', 'day': 'd0'}, {'user': 'u55', 'country': 'IN', 'day': 'd1'}, {'user': 'u56', 'country': 'IN', 'day': 'd2'}, {'user': 'u57', 'country': 'IN', 'day': 'd0'}, {'user': 'u58', 'country': 'US', 'day': 'd1'}, {'user': 'u59', 'country': 'US', 'day': 'd2'}, {'user': 'u60', 'country': 'IN', 'day': 'd0'}, {'user': 'u61', 'country': 'IN', 'day': 'd1'}, {'user': 'u62', 'country': 'IN', 'day': 'd2'}, {'user': 'u63', 'country': 'IN', 'day': 'd0'}, {'user': 'u64', 'country': 'IN', 'day': 'd1'}, {'user': 'u65', 'country': 'IN', 'day': 'd2'}, {'user': 'u66', 'country': 'IN', 'day': 'd0'}, {'user': 'u67', 'country': 'IN', 'day': 'd1'}, {'user': 'u68', 'country': 'US', 'day': 'd2'}, {'user': 'u69', 'country': 'US', 'day': 'd0'}, {'user': 'u70', 'country': 'IN', 'day': 'd1'}, {'user': 'u71', 'country': 'IN', 'day': 'd2'}, {'user': 'u72', 'country': 'IN', 'day': 'd0'}, {'user': 'u73', 'country': 'IN', 'day': 'd1'}, {'user': 'u74', 'country': 'IN', 'day': 'd2'}, {'user': 'u75', 'country': 'IN', 'day': 'd0'}, {'user': 'u76', 'country': 'IN', 'day': 'd1'}, {'user': 'u77', 'country': 'IN', 'day': 'd2'}, {'user': 'u78', 'country': 'US', 'day': 'd0'}, {'user': 'u79', 'country': 'US', 'day': 'd1'}, {'user': 'u80', 'country': 'IN', 'day': 'd2'}, {'user': 'u81', 'country': 'IN', 'day': 'd0'}, {'user': 'u82', 'country': 'IN', 'day': 'd1'}, {'user': 'u83', 'country': 'IN', 'day': 'd2'}, {'user': 'u84', 'country': 'IN', 'day': 'd0'}, {'user': 'u85', 'country': 'IN', 'day': 'd1'}, {'user': 'u86', 'country': 'IN', 'day': 'd2'}, {'user': 'u87', 'country': 'IN', 'day': 'd0'}, {'user': 'u88', 'country': 'US', 'day': 'd1'}, {'user': 'u89', 'country': 'US', 'day': 'd2'}, {'user': 'u90', 'country': 'IN', 'day': 'd0'}, {'user': 'u91', 'country': 'IN', 'day': 'd1'}, {'user': 'u92', 'country': 'IN', 'day': 'd2'}, {'user': 'u93', 'country': 'IN', 'day': 'd0'}, {'user': 'u94', 'country': 'IN', 'day': 'd1'}, {'user': 'u95', 'country': 'IN', 'day': 'd2'}, {'user': 'u96', 'country': 'IN', 'day': 'd0'}, {'user': 'u97', 'country': 'IN', 'day': 'd1'}, {'user': 'u98', 'country': 'US', 'day': 'd2'}, {'user': 'u99', 'country': 'US', 'day': 'd0'}, {'user': 'u100', 'country': 'IN', 'day': 'd1'}, {'user': 'u101', 'country': 'IN', 'day': 'd2'}, {'user': 'u102', 'country': 'IN', 'day': 'd0'}, {'user': 'u103', 'country': 'IN', 'day': 'd1'}, {'user': 'u104', 'country': 'IN', 'day': 'd2'}, {'user': 'u105', 'country': 'IN', 'day': 'd0'}, {'user': 'u106', 'country': 'IN', 'day': 'd1'}, {'user': 'u107', 'country': 'IN', 'day': 'd2'}, {'user': 'u108', 'country': 'US', 'day': 'd0'}, {'user': 'u109', 'country': 'US', 'day': 'd1'}, {'user': 'u110', 'country': 'IN', 'day': 'd2'}, {'user': 'u111', 'country': 'IN', 'day': 'd0'}, {'user': 'u112', 'country': 'IN', 'day': 'd1'}, {'user': 'u113', 'country': 'IN', 'day': 'd2'}, {'user': 'u114', 'country': 'IN', 'day': 'd0'}, {'user': 'u115', 'country': 'IN', 'day': 'd1'}, {'user': 'u116', 'country': 'IN', 'day': 'd2'}, {'user': 'u117', 'country': 'IN', 'day': 'd0'}, {'user': 'u118', 'country': 'US', 'day': 'd1'}, {'user': 'u119', 'country': 'US', 'day': 'd2'}, {'user': 'u120', 'country': 'IN', 'day': 'd0'}, {'user': 'u121', 'country': 'IN', 'day': 'd1'}, {'user': 'u122', 'country': 'IN', 'day': 'd2'}, {'user': 'u123', 'country': 'IN', 'day': 'd0'}, {'user': 'u124', 'country': 'IN', 'day': 'd1'}, {'user': 'u125', 'country': 'IN', 'day': 'd2'}, {'user': 'u126', 'country': 'IN', 'day': 'd0'}, {'user': 'u127', 'country': 'IN', 'day': 'd1'}, {'user': 'u128', 'country': 'US', 'day': 'd2'}, {'user': 'u129', 'country': 'US', 'day': 'd0'}, {'user': 'u130', 'country': 'IN', 'day': 'd1'}, {'user': 'u131', 'country': 'IN', 'day': 'd2'}, {'user': 'u132', 'country': 'IN', 'day': 'd0'}, {'user': 'u133', 'country': 'IN', 'day': 'd1'}, {'user': 'u134', 'country': 'IN', 'day': 'd2'}, {'user': 'u135', 'country': 'IN', 'day': 'd0'}, {'user': 'u136', 'country': 'IN', 'day': 'd1'}, {'user': 'u137', 'country': 'IN', 'day': 'd2'}, {'user': 'u138', 'country': 'US', 'day': 'd0'}, {'user': 'u139', 'country': 'US', 'day': 'd1'}, {'user': 'u140', 'country': 'IN', 'day': 'd2'}, {'user': 'u141', 'country': 'IN', 'day': 'd0'}, {'user': 'u142', 'country': 'IN', 'day': 'd1'}, {'user': 'u143', 'country': 'IN', 'day': 'd2'}, {'user': 'u144', 'country': 'IN', 'day': 'd0'}, {'user': 'u145', 'country': 'IN', 'day': 'd1'}, {'user': 'u146', 'country': 'IN', 'day': 'd2'}, {'user': 'u147', 'country': 'IN', 'day': 'd0'}, {'user': 'u148', 'country': 'US', 'day': 'd1'}, {'user': 'u149', 'country': 'US', 'day': 'd2'}, {'user': 'u150', 'country': 'IN', 'day': 'd0'}, {'user': 'u151', 'country': 'IN', 'day': 'd1'}, {'user': 'u152', 'country': 'IN', 'day': 'd2'}, {'user': 'u153', 'country': 'IN', 'day': 'd0'}, {'user': 'u154', 'country': 'IN', 'day': 'd1'}, {'user': 'u155', 'country': 'IN', 'day': 'd2'}, {'user': 'u156', 'country': 'IN', 'day': 'd0'}, {'user': 'u157', 'country': 'IN', 'day': 'd1'}, {'user': 'u158', 'country': 'US', 'day': 'd2'}, {'user': 'u159', 'country': 'US', 'day': 'd0'}, {'user': 'u160', 'country': 'IN', 'day': 'd1'}, {'user': 'u161', 'country': 'IN', 'day': 'd2'}, {'user': 'u162', 'country': 'IN', 'day': 'd0'}, {'user': 'u163', 'country': 'IN', 'day': 'd1'}, {'user': 'u164', 'country': 'IN', 'day': 'd2'}, {'user': 'u165', 'country': 'IN', 'day': 'd0'}, {'user': 'u166', 'country': 'IN', 'day': 'd1'}, {'user': 'u167', 'country': 'IN', 'day': 'd2'}, {'user': 'u168', 'country': 'US', 'day': 'd0'}, {'user': 'u169', 'country': 'US', 'day': 'd1'}, {'user': 'u170', 'country': 'IN', 'day': 'd2'}, {'user': 'u171', 'country': 'IN', 'day': 'd0'}, {'user': 'u172', 'country': 'IN', 'day': 'd1'}, {'user': 'u173', 'country': 'IN', 'day': 'd2'}, {'user': 'u174', 'country': 'IN', 'day': 'd0'}, {'user': 'u175', 'country': 'IN', 'day': 'd1'}, {'user': 'u176', 'country': 'IN', 'day': 'd2'}, {'user': 'u177', 'country': 'IN', 'day': 'd0'}, {'user': 'u178', 'country': 'US', 'day': 'd1'}, {'user': 'u179', 'country': 'US', 'day': 'd2'}, {'user': 'u180', 'country': 'IN', 'day': 'd0'}, {'user': 'u181', 'country': 'IN', 'day': 'd1'}, {'user': 'u182', 'country': 'IN', 'day': 'd2'}, {'user': 'u183', 'country': 'IN', 'day': 'd0'}, {'user': 'u184', 'country': 'IN', 'day': 'd1'}, {'user': 'u185', 'country': 'IN', 'day': 'd2'}, {'user': 'u186', 'country': 'IN', 'day': 'd0'}, {'user': 'u187', 'country': 'IN', 'day': 'd1'}, {'user': 'u188', 'country': 'US', 'day': 'd2'}, {'user': 'u189', 'country': 'US', 'day': 'd0'}, {'user': 'u190', 'country': 'IN', 'day': 'd1'}, {'user': 'u191', 'country': 'IN', 'day': 'd2'}, {'user': 'u192', 'country': 'IN', 'day': 'd0'}, {'user': 'u193', 'country': 'IN', 'day': 'd1'}, {'user': 'u194', 'country': 'IN', 'day': 'd2'}, {'user': 'u195', 'country': 'IN', 'day': 'd0'}, {'user': 'u196', 'country': 'IN', 'day': 'd1'}, {'user': 'u197', 'country': 'IN', 'day': 'd2'}, {'user': 'u198', 'country': 'US', 'day': 'd0'}, {'user': 'u199', 'country': 'US', 'day': 'd1'}], candidates = ['user', 'country', 'day'], n = 4` → `({'user': 25.0, 'country': 100.0, 'day': 66.5}, 'user')`

In [ ]:
def shard_key(records, candidates, n):
    """
    Hash-shard records on each candidate field into n shards. Return the largest
    shard's share (%, rounded to 1) for each candidate, and the candidate with
    the smallest maximum.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shard_key, [
    (([{'user': 'u0', 'country': 'IN', 'day': 'd0'}, {'user': 'u1', 'country': 'IN', 'day': 'd1'}, {'user': 'u2', 'country': 'IN', 'day': 'd2'}, {'user': 'u3', 'country': 'IN', 'day': 'd0'}, {'user': 'u4', 'country': 'IN', 'day': 'd1'}, {'user': 'u5', 'country': 'IN', 'day': 'd2'}, {'user': 'u6', 'country': 'IN', 'day': 'd0'}, {'user': 'u7', 'country': 'IN', 'day': 'd1'}, {'user': 'u8', 'country': 'US', 'day': 'd2'}, {'user': 'u9', 'country': 'US', 'day': 'd0'}, {'user': 'u10', 'country': 'IN', 'day': 'd1'}, {'user': 'u11', 'country': 'IN', 'day': 'd2'}, {'user': 'u12', 'country': 'IN', 'day': 'd0'}, {'user': 'u13', 'country': 'IN', 'day': 'd1'}, {'user': 'u14', 'country': 'IN', 'day': 'd2'}, {'user': 'u15', 'country': 'IN', 'day': 'd0'}, {'user': 'u16', 'country': 'IN', 'day': 'd1'}, {'user': 'u17', 'country': 'IN', 'day': 'd2'}, {'user': 'u18', 'country': 'US', 'day': 'd0'}, {'user': 'u19', 'country': 'US', 'day': 'd1'}, {'user': 'u20', 'country': 'IN', 'day': 'd2'}, {'user': 'u21', 'country': 'IN', 'day': 'd0'}, {'user': 'u22', 'country': 'IN', 'day': 'd1'}, {'user': 'u23', 'country': 'IN', 'day': 'd2'}, {'user': 'u24', 'country': 'IN', 'day': 'd0'}, {'user': 'u25', 'country': 'IN', 'day': 'd1'}, {'user': 'u26', 'country': 'IN', 'day': 'd2'}, {'user': 'u27', 'country': 'IN', 'day': 'd0'}, {'user': 'u28', 'country': 'US', 'day': 'd1'}, {'user': 'u29', 'country': 'US', 'day': 'd2'}, {'user': 'u30', 'country': 'IN', 'day': 'd0'}, {'user': 'u31', 'country': 'IN', 'day': 'd1'}, {'user': 'u32', 'country': 'IN', 'day': 'd2'}, {'user': 'u33', 'country': 'IN', 'day': 'd0'}, {'user': 'u34', 'country': 'IN', 'day': 'd1'}, {'user': 'u35', 'country': 'IN', 'day': 'd2'}, {'user': 'u36', 'country': 'IN', 'day': 'd0'}, {'user': 'u37', 'country': 'IN', 'day': 'd1'}, {'user': 'u38', 'country': 'US', 'day': 'd2'}, {'user': 'u39', 'country': 'US', 'day': 'd0'}, {'user': 'u40', 'country': 'IN', 'day': 'd1'}, {'user': 'u41', 'country': 'IN', 'day': 'd2'}, {'user': 'u42', 'country': 'IN', 'day': 'd0'}, {'user': 'u43', 'country': 'IN', 'day': 'd1'}, {'user': 'u44', 'country': 'IN', 'day': 'd2'}, {'user': 'u45', 'country': 'IN', 'day': 'd0'}, {'user': 'u46', 'country': 'IN', 'day': 'd1'}, {'user': 'u47', 'country': 'IN', 'day': 'd2'}, {'user': 'u48', 'country': 'US', 'day': 'd0'}, {'user': 'u49', 'country': 'US', 'day': 'd1'}, {'user': 'u50', 'country': 'IN', 'day': 'd2'}, {'user': 'u51', 'country': 'IN', 'day': 'd0'}, {'user': 'u52', 'country': 'IN', 'day': 'd1'}, {'user': 'u53', 'country': 'IN', 'day': 'd2'}, {'user': 'u54', 'country': 'IN', 'day': 'd0'}, {'user': 'u55', 'country': 'IN', 'day': 'd1'}, {'user': 'u56', 'country': 'IN', 'day': 'd2'}, {'user': 'u57', 'country': 'IN', 'day': 'd0'}, {'user': 'u58', 'country': 'US', 'day': 'd1'}, {'user': 'u59', 'country': 'US', 'day': 'd2'}, {'user': 'u60', 'country': 'IN', 'day': 'd0'}, {'user': 'u61', 'country': 'IN', 'day': 'd1'}, {'user': 'u62', 'country': 'IN', 'day': 'd2'}, {'user': 'u63', 'country': 'IN', 'day': 'd0'}, {'user': 'u64', 'country': 'IN', 'day': 'd1'}, {'user': 'u65', 'country': 'IN', 'day': 'd2'}, {'user': 'u66', 'country': 'IN', 'day': 'd0'}, {'user': 'u67', 'country': 'IN', 'day': 'd1'}, {'user': 'u68', 'country': 'US', 'day': 'd2'}, {'user': 'u69', 'country': 'US', 'day': 'd0'}, {'user': 'u70', 'country': 'IN', 'day': 'd1'}, {'user': 'u71', 'country': 'IN', 'day': 'd2'}, {'user': 'u72', 'country': 'IN', 'day': 'd0'}, {'user': 'u73', 'country': 'IN', 'day': 'd1'}, {'user': 'u74', 'country': 'IN', 'day': 'd2'}, {'user': 'u75', 'country': 'IN', 'day': 'd0'}, {'user': 'u76', 'country': 'IN', 'day': 'd1'}, {'user': 'u77', 'country': 'IN', 'day': 'd2'}, {'user': 'u78', 'country': 'US', 'day': 'd0'}, {'user': 'u79', 'country': 'US', 'day': 'd1'}, {'user': 'u80', 'country': 'IN', 'day': 'd2'}, {'user': 'u81', 'country': 'IN', 'day': 'd0'}, {'user': 'u82', 'country': 'IN', 'day': 'd1'}, {'user': 'u83', 'country': 'IN', 'day': 'd2'}, {'user': 'u84', 'country': 'IN', 'day': 'd0'}, {'user': 'u85', 'country': 'IN', 'day': 'd1'}, {'user': 'u86', 'country': 'IN', 'day': 'd2'}, {'user': 'u87', 'country': 'IN', 'day': 'd0'}, {'user': 'u88', 'country': 'US', 'day': 'd1'}, {'user': 'u89', 'country': 'US', 'day': 'd2'}, {'user': 'u90', 'country': 'IN', 'day': 'd0'}, {'user': 'u91', 'country': 'IN', 'day': 'd1'}, {'user': 'u92', 'country': 'IN', 'day': 'd2'}, {'user': 'u93', 'country': 'IN', 'day': 'd0'}, {'user': 'u94', 'country': 'IN', 'day': 'd1'}, {'user': 'u95', 'country': 'IN', 'day': 'd2'}, {'user': 'u96', 'country': 'IN', 'day': 'd0'}, {'user': 'u97', 'country': 'IN', 'day': 'd1'}, {'user': 'u98', 'country': 'US', 'day': 'd2'}, {'user': 'u99', 'country': 'US', 'day': 'd0'}, {'user': 'u100', 'country': 'IN', 'day': 'd1'}, {'user': 'u101', 'country': 'IN', 'day': 'd2'}, {'user': 'u102', 'country': 'IN', 'day': 'd0'}, {'user': 'u103', 'country': 'IN', 'day': 'd1'}, {'user': 'u104', 'country': 'IN', 'day': 'd2'}, {'user': 'u105', 'country': 'IN', 'day': 'd0'}, {'user': 'u106', 'country': 'IN', 'day': 'd1'}, {'user': 'u107', 'country': 'IN', 'day': 'd2'}, {'user': 'u108', 'country': 'US', 'day': 'd0'}, {'user': 'u109', 'country': 'US', 'day': 'd1'}, {'user': 'u110', 'country': 'IN', 'day': 'd2'}, {'user': 'u111', 'country': 'IN', 'day': 'd0'}, {'user': 'u112', 'country': 'IN', 'day': 'd1'}, {'user': 'u113', 'country': 'IN', 'day': 'd2'}, {'user': 'u114', 'country': 'IN', 'day': 'd0'}, {'user': 'u115', 'country': 'IN', 'day': 'd1'}, {'user': 'u116', 'country': 'IN', 'day': 'd2'}, {'user': 'u117', 'country': 'IN', 'day': 'd0'}, {'user': 'u118', 'country': 'US', 'day': 'd1'}, {'user': 'u119', 'country': 'US', 'day': 'd2'}, {'user': 'u120', 'country': 'IN', 'day': 'd0'}, {'user': 'u121', 'country': 'IN', 'day': 'd1'}, {'user': 'u122', 'country': 'IN', 'day': 'd2'}, {'user': 'u123', 'country': 'IN', 'day': 'd0'}, {'user': 'u124', 'country': 'IN', 'day': 'd1'}, {'user': 'u125', 'country': 'IN', 'day': 'd2'}, {'user': 'u126', 'country': 'IN', 'day': 'd0'}, {'user': 'u127', 'country': 'IN', 'day': 'd1'}, {'user': 'u128', 'country': 'US', 'day': 'd2'}, {'user': 'u129', 'country': 'US', 'day': 'd0'}, {'user': 'u130', 'country': 'IN', 'day': 'd1'}, {'user': 'u131', 'country': 'IN', 'day': 'd2'}, {'user': 'u132', 'country': 'IN', 'day': 'd0'}, {'user': 'u133', 'country': 'IN', 'day': 'd1'}, {'user': 'u134', 'country': 'IN', 'day': 'd2'}, {'user': 'u135', 'country': 'IN', 'day': 'd0'}, {'user': 'u136', 'country': 'IN', 'day': 'd1'}, {'user': 'u137', 'country': 'IN', 'day': 'd2'}, {'user': 'u138', 'country': 'US', 'day': 'd0'}, {'user': 'u139', 'country': 'US', 'day': 'd1'}, {'user': 'u140', 'country': 'IN', 'day': 'd2'}, {'user': 'u141', 'country': 'IN', 'day': 'd0'}, {'user': 'u142', 'country': 'IN', 'day': 'd1'}, {'user': 'u143', 'country': 'IN', 'day': 'd2'}, {'user': 'u144', 'country': 'IN', 'day': 'd0'}, {'user': 'u145', 'country': 'IN', 'day': 'd1'}, {'user': 'u146', 'country': 'IN', 'day': 'd2'}, {'user': 'u147', 'country': 'IN', 'day': 'd0'}, {'user': 'u148', 'country': 'US', 'day': 'd1'}, {'user': 'u149', 'country': 'US', 'day': 'd2'}, {'user': 'u150', 'country': 'IN', 'day': 'd0'}, {'user': 'u151', 'country': 'IN', 'day': 'd1'}, {'user': 'u152', 'country': 'IN', 'day': 'd2'}, {'user': 'u153', 'country': 'IN', 'day': 'd0'}, {'user': 'u154', 'country': 'IN', 'day': 'd1'}, {'user': 'u155', 'country': 'IN', 'day': 'd2'}, {'user': 'u156', 'country': 'IN', 'day': 'd0'}, {'user': 'u157', 'country': 'IN', 'day': 'd1'}, {'user': 'u158', 'country': 'US', 'day': 'd2'}, {'user': 'u159', 'country': 'US', 'day': 'd0'}, {'user': 'u160', 'country': 'IN', 'day': 'd1'}, {'user': 'u161', 'country': 'IN', 'day': 'd2'}, {'user': 'u162', 'country': 'IN', 'day': 'd0'}, {'user': 'u163', 'country': 'IN', 'day': 'd1'}, {'user': 'u164', 'country': 'IN', 'day': 'd2'}, {'user': 'u165', 'country': 'IN', 'day': 'd0'}, {'user': 'u166', 'country': 'IN', 'day': 'd1'}, {'user': 'u167', 'country': 'IN', 'day': 'd2'}, {'user': 'u168', 'country': 'US', 'day': 'd0'}, {'user': 'u169', 'country': 'US', 'day': 'd1'}, {'user': 'u170', 'country': 'IN', 'day': 'd2'}, {'user': 'u171', 'country': 'IN', 'day': 'd0'}, {'user': 'u172', 'country': 'IN', 'day': 'd1'}, {'user': 'u173', 'country': 'IN', 'day': 'd2'}, {'user': 'u174', 'country': 'IN', 'day': 'd0'}, {'user': 'u175', 'country': 'IN', 'day': 'd1'}, {'user': 'u176', 'country': 'IN', 'day': 'd2'}, {'user': 'u177', 'country': 'IN', 'day': 'd0'}, {'user': 'u178', 'country': 'US', 'day': 'd1'}, {'user': 'u179', 'country': 'US', 'day': 'd2'}, {'user': 'u180', 'country': 'IN', 'day': 'd0'}, {'user': 'u181', 'country': 'IN', 'day': 'd1'}, {'user': 'u182', 'country': 'IN', 'day': 'd2'}, {'user': 'u183', 'country': 'IN', 'day': 'd0'}, {'user': 'u184', 'country': 'IN', 'day': 'd1'}, {'user': 'u185', 'country': 'IN', 'day': 'd2'}, {'user': 'u186', 'country': 'IN', 'day': 'd0'}, {'user': 'u187', 'country': 'IN', 'day': 'd1'}, {'user': 'u188', 'country': 'US', 'day': 'd2'}, {'user': 'u189', 'country': 'US', 'day': 'd0'}, {'user': 'u190', 'country': 'IN', 'day': 'd1'}, {'user': 'u191', 'country': 'IN', 'day': 'd2'}, {'user': 'u192', 'country': 'IN', 'day': 'd0'}, {'user': 'u193', 'country': 'IN', 'day': 'd1'}, {'user': 'u194', 'country': 'IN', 'day': 'd2'}, {'user': 'u195', 'country': 'IN', 'day': 'd0'}, {'user': 'u196', 'country': 'IN', 'day': 'd1'}, {'user': 'u197', 'country': 'IN', 'day': 'd2'}, {'user': 'u198', 'country': 'US', 'day': 'd0'}, {'user': 'u199', 'country': 'US', 'day': 'd1'}], ['user', 'country', 'day'], 4), ({'user': 25.0, 'country': 100.0, 'day': 66.5}, 'user')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count shard sizes per candidate.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter` per candidate and `min`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count shard sizes per candidate.

```python
def shard_key(records, candidates, n):
    shares = {}
    for c in candidates:
        counts = [0] * n
        for r in records:
            counts[zlib.crc32(str(r[c]).encode()) % n] += 1
        shares[c] = round(max(counts) / len(records) * 100, 1)
    best = candidates[0]
    for c in candidates:
        if shares[c] < shares[best]:
            best = c
    return (shares, best)
```

**Cost:** O(c · r) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter` per candidate and `min`.

```python
def shard_key(records, candidates, n):
    shares = {c: round(max(Counter(zlib.crc32(str(r[c]).encode()) % n for r in records).values()) / len(records) * 100, 1)
              for c in candidates}
    return (shares, min(candidates, key=shares.get))
```

**Cost:** O(c · r) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count shard sizes per candidate. That costs O(c · r) time and O(n) space."

"I can do better: `Counter` per candidate and `min`. That's O(c · r) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/choose-a-shard-key)

<a id="q18"></a>
### Q18. Split Oversized Shards

🟡 Interview standard · *Resharding by halves*

Shards are sorted key lists. Split any shard larger than `max_size` in half (left gets the smaller half), repeating until all fit. Return the new shard sizes in key order.

**Examples**

- `shards = [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12], [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]], max_size = 4` → `[2, 3, 2, 3, 3, 4, 4, 4, 2, 3]`

In [ ]:
def split_shards(shards, max_size):
    """
    Shards are sorted key lists. Split any shard larger than max_size in half
    (left gets the smaller half), repeating until all fit. Return the new shard
    sizes in key order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_shards, [
    (([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12], [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]], 4), [2, 3, 2, 3, 3, 4, 4, 4, 2, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop until no shard is too big.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Recursive split per shard.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop until no shard is too big.

```python
def split_shards(shards, max_size):
    shards = [list(s) for s in shards]
    changed = True
    while changed:
        changed = False
        out = []
        for s in shards:
            if len(s) > max_size:
                out += [s[:len(s) // 2], s[len(s) // 2:]]
                changed = True
            else:
                out.append(s)
        shards = out
    return [len(s) for s in shards]
```

**Cost:** O(n · s) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Recursive split per shard.

```python
def split_shards(shards, max_size):
    def sizes(n):
        return [n] if n <= max_size else sizes(n // 2) + sizes(n - n // 2)
    return [x for s in shards for x in sizes(len(s))]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop until no shard is too big. That costs O(n · s) time and O(n) space."

"I can do better: Recursive split per shard. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `shards = [[0, 1, 2, 3, 4, 5]], max_size = 2`

1. sizes() returns [1]
2. sizes() returns [2]
3. sizes() returns [1, 2]
4. sizes() returns [1]
5. sizes() returns [2]
6. sizes() returns [1, 2]
7. sizes() returns [1, 2, 1, 2]
8. split\_shards() returns [1, 2, 1, 2]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/split-oversized-shards)

<a id="q19"></a>
### Q19. Global Top-K Across Shards

🟡 Interview standard · *Merge sorted results*

Each shard returns its scores sorted from high to low. Return the global top `k`.

**Examples**

- `shards = [[98, 90, 75, 60], [99, 70, 65], [95, 94, 93, 10]], k = 5` → `[99, 98, 95, 94, 93]`

In [ ]:
import heapq

from itertools import islice

def top_k(shards, k):
    """
    Each shard returns its scores sorted from high to low. Return the global top
    k.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_k, [
    (([[98, 90, 75, 60], [99, 70, 65], [95, 94, 93, 10]], 5), [99, 98, 95, 94, 93]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Concatenate and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`heapq.merge(..., reverse=True)` and stop after k.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Concatenate and sort.

```python
def top_k(shards, k):
    return sorted([x for s in shards for x in s], reverse=True)[:k]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `heapq.merge(..., reverse=True)` and stop after k.

```python
def top_k(shards, k):
    return list(islice(heapq.merge(*shards, reverse=True), k))
```

**Cost:** O(k log s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Concatenate and sort. That costs O(n log n) time and O(n) space."

"I can do better: `heapq.merge(..., reverse=True)` and stop after k. That's O(k log s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/global-top-k-across-shards)

<a id="q20"></a>
### Q20. The Lost Update

🟡 Interview standard · *Read-modify-write races*

Two transactions each read a counter into a local copy, then write back copy + delta. Run the interleaved `schedule` of `(tx, 'read')` and `(tx, 'write', delta)`. Return `(final value with read-modify-write, final value with atomic increments)`.

**Examples**

- `start = 100, schedule = [('T1', 'read'), ('T2', 'read'), ('T1', 'write', 10), ('T2', 'write', -30)]` → `(70, 80)`
- `start = 100, schedule = [('T1', 'read'), ('T1', 'write', 10), ('T2', 'read'), ('T2', 'write', -30)]` → `(80, 80)`

In [ ]:
def lost_update(start, schedule):
    """
    Two transactions each read a counter into a local copy, then write back copy
    + delta. Run the interleaved schedule of (tx, 'read') and (tx, 'write',
    delta). Return (final value with read-modify-write, final value with atomic
    increments).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lost_update, [
    ((100, [('T1', 'read'), ('T2', 'read'), ('T1', 'write', 10), ('T2', 'write', -30)]), (70, 80)),
    ((100, [('T1', 'read'), ('T1', 'write', 10), ('T2', 'read'), ('T2', 'write', -30)]), (80, 80)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate with a dict of local copies.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same; atomic = start + sum of deltas.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate with a dict of local copies.

```python
def lost_update(start, schedule):
    value, local = start, {}
    for step in schedule:
        if step[1] == "read":
            local[step[0]] = value
        else:
            value = local[step[0]] + step[2]
    atomic = start
    for step in schedule:
        if step[1] == "write":
            atomic += step[2]
    return (value, atomic)
```

**Cost:** O(s) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same; atomic = start + sum of deltas.

```python
def lost_update(start, schedule):
    value, local = start, {}
    for tx, op, *delta in schedule:
        if op == "read":
            local[tx] = value
        else:
            value = local[tx] + delta[0]
    return (value, start + sum(s[2] for s in schedule if s[1] == "write"))
```

**Cost:** O(s) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate with a dict of local copies. That costs O(s) time and O(t) space."

"I can do better: Same; atomic = start + sum of deltas. That's O(s) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/databases/the-lost-update)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. SQL or NoSQL?</b></summary>

Default to SQL for structured data with relationships and transactions. Choose NoSQL for a clear reason: huge write throughput, simple key access at massive scale, flexible documents, or a specialised model (graph, search, time-series).

</details>

<details>
<summary><b>2. How do you scale a database?</b></summary>

In order: indexes and query fixes, caching, read replicas, vertical scaling, then partitioning or sharding — and archive old data. Sharding is last because it complicates joins, transactions and operations.

</details>

<details>
<summary><b>3. What is replication lag and how do you handle it?</b></summary>

The delay before a replica applies the primary's writes. Route a user's reads to the primary right after they write, use monotonic reads (same replica), or wait for a replication position before reading.

</details>

<details>
<summary><b>4. What makes a good shard key?</b></summary>

High cardinality, even distribution of data and traffic, and present in most queries. Avoid monotonically increasing keys with range sharding and low-cardinality keys like country.

</details>

<details>
<summary><b>5. B-tree or LSM tree?</b></summary>

B-trees suit read-heavy workloads with in-place updates; LSM trees suit write-heavy workloads, trading some read cost and background compaction for fast sequential writes.

</details>

<details>
<summary><b>6. How do you do a transaction across shards?</b></summary>

Avoid it by choosing a shard key that keeps related rows together. Otherwise use two-phase commit (blocking, slow) or a saga of local transactions with compensations (covered in the consistency topic).

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Choosing a database
- [ ] I can explain: Indexes and storage engines
- [ ] I can explain: Replication
- [ ] I can explain: Sharding (partitioning)
- [ ] I can explain: Transactions and concurrency

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Range Sharding and the Hot Shard
- [ ] [Q17](#q17) Choose a Shard Key
- [ ] [Q18](#q18) Split Oversized Shards
- [ ] [Q19](#q19) Global Top-K Across Shards
- [ ] [Q20](#q20) The Lost Update

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.